In [ ]:
# Estructura esperada (relativa a la raíz del repo):
#   ./01_data/<proyecto>/AOI/                -> polígonos AOI (incluidos)
#   ./01_data/<proyecto>/images/             -> imágenes Planet (NO incluidas; bajo licencia Planet)
#   ./03_figures_&_results/<proyecto>/        -> tablas derivadas y figuras (salidas)

# Section 0 — Construcción de las bases de píxeles (portado de 02_1)

Este bloque hace **autocontenido** a este notebook: genera los dos parquets que necesita el análisis (`Pixel_Spectral_Master_v2.parquet` y `Pixel_Availability_QC.parquet`) para cada proyecto.

- Si los parquets **ya existen** (caso normal en tu máquina), el *guard* de caché los detecta y **no reconstruye nada** (la carga de la Sección 0-bis de abajo los lee tal cual: comportamiento idéntico al de antes).
- Si **faltan**, se reconstruyen desde los TIFs/XML/shapefiles crudos para ambos proyectos.


### 0.1 Configuración de construcción (rutas crudas, heredada de 02_1)

In [ ]:
# ==============================================================================
# 1. PROJECT CONFIGURATION
# Planet-UAV Reliability Assessment Framework
# ==============================================================================

PROJECTS = {

    "avocado_mango": {

        "project_name": "avocado_mango",

        # UAV before Planet time-series
        "temporal_mode": "forward",

        "uav_date": "2019-08-16",

        "output_dir": r".\03_figures_&_results\02_avocado_mango",

        "aois": {

            "avocado": {

                "img_dir": r".\01_data\02_avocado_mango\images\nubes_20_harmonized",

                "shp": r".\01_data\02_avocado_mango\AOI\avocado_group.shp"
            },

            "mango": {

                "img_dir": r".\01_data\02_avocado_mango\images\nubes_20_harmonized",

                "shp": r".\01_data\02_avocado_mango\AOI\mangoe_group.shp"
            }
        }
    },


    "corema_album": {

        "project_name": "corema_album",

        # UAV after Planet time-series
        "temporal_mode": "backward",

        "uav_date": "2025-06-20",

        "output_dir": r".\03_figures_&_results\01_corema_album",

        "aois": {

            "corema_A": {

                "img_dir": r".\01_data\01_corema_album\images\nubes_20_harmonized",

                "shp": r".\01_data\01_corema_album\AOI\veg_AOI_1.shp"
            },

            "corema_B": {

                "img_dir": r".\01_data\01_corema_album\images\nubes_v2_20_harmonized",

                "shp": r".\01_data\01_corema_album\AOI\veg_AOI_2.shp"
            }
        }
    }
}


# ==============================================================================
# SELECT ACTIVE EXPERIMENT
# ==============================================================================

PROJECT_CONFIG = PROJECTS["avocado_mango"]
# PROJECT_CONFIG = PROJECTS["corema_album"]


print(
    f"""
==================================================
ACTIVE PROJECT
==================================================
Project: {PROJECT_CONFIG['project_name']}
Mode:    {PROJECT_CONFIG['temporal_mode']}
UAV:     {PROJECT_CONFIG['uav_date']}
AOIs:    {list(PROJECT_CONFIG['aois'].keys())}
==================================================
"""
)

### 0.2 Función de construcción (celdas 1,2,4,5,6,7,11 de 02_1, verbatim, envueltas con guard de caché)

In [ ]:
def build_project_caches(PROJECT_CONFIG):
    # ---- imports defensivos (las celdas también re-importan) ----
    import os, glob, numpy as np, pandas as pd
    import rasterio, geopandas as gpd
    from rasterio.features import geometry_mask, rasterize
    import xml.etree.ElementTree as ET
    from shapely.geometry import box
    try:
        from tqdm import tqdm
    except Exception:
        def tqdm(x, *a, **k): return x
    # ---- guard de caché: si ya existen los dos parquets, no reconstruir ----
    _spec = os.path.join(PROJECT_CONFIG['output_dir'], 'Pixel_Spectral_Master_v2.parquet')
    _qc   = os.path.join(PROJECT_CONFIG['output_dir'], 'Pixel_Availability_QC.parquet')
    if os.path.exists(_spec) and os.path.exists(_qc):
        print(f"[CACHE OK] {PROJECT_CONFIG.get('project_name','?')}: parquets ya existen; no se reconstruye.")
        return
    print(f"[BUILD] {PROJECT_CONFIG.get('project_name','?')}: construyendo desde TIFs crudos (puede tardar)...")
    # ===== [02_1 celda 1] =====
    # ==============================================================================
    # 2. PLANET IMAGE INVENTORY BUILDER
    # ==============================================================================
    # Objective:
    # Generate a standardized AOI x Planet acquisition database.
    #
    # Output:
    # df_inventory
    #
    # One row = one Planet observation available for one vegetation target.
    # ==============================================================================


    import os
    import glob
    import pandas as pd
    from datetime import datetime


    # ------------------------------------------------------------------------------
    # Helper functions
    # ------------------------------------------------------------------------------

    def find_planet_files(img_dir):
        """
        Locate only valid PlanetScope spectral images and XML metadata.
    
        Excludes:
        - UDM masks
        - quality masks
        - auxiliary rasters
        """

        all_tifs = glob.glob(
            os.path.join(
                img_dir,
                "**",
                "*.tif"
            ),
            recursive=True
        )


        tif_files = [
            f for f in all_tifs
        
            # Keep only spectral products
            if (
                ("AnalyticMS" in os.path.basename(f))
                and ("udm" not in os.path.basename(f).lower())
            )
        ]


        xml_files = glob.glob(
            os.path.join(
                img_dir,
                "**",
                "*.xml"
            ),
            recursive=True
        )


        return tif_files, xml_files



    def extract_planet_id(filepath):
        """
        Extract Planet unique acquisition ID.
    
        Example:
        20230116_104003_14_247b_3B_AnalyticMS_SR_8b.tif
    
        returns:
        20230116_104003_14_247b
        """

        filename = os.path.basename(filepath)

        parts = filename.split("_")

        smart_id = "_".join(parts[:4])

        return smart_id



    def extract_datetime_from_id(smart_id):
        """
        Extract complete Planet acquisition datetime (UTC).

        Example:
        20230116_104003_14_247b

        returns:
        2023-01-16 10:40:03 UTC
        """

        parts = smart_id.split("_")

        datetime_string = (
            parts[0]
            +
            parts[1]
        )

        return pd.to_datetime(
            datetime_string,
            format="%Y%m%d%H%M%S",
            utc=True
        )



    def match_xml(tif_path, xml_files):
        """
        Match Planet image with metadata XML.
        """

        smart_id = extract_planet_id(tif_path)

        matches = [
            x for x in xml_files
            if smart_id in os.path.basename(x)
        ]

        if len(matches) > 0:
            return matches[0]

        return None



    # ------------------------------------------------------------------------------
    # Build inventory
    # ------------------------------------------------------------------------------

    records = []


    for aoi_name, cfg in PROJECT_CONFIG["aois"].items():

        print(f"\nProcessing AOI: {aoi_name}")


        tif_files, xml_files = find_planet_files(
            cfg["img_dir"]
        )


        print(f"  TIF found: {len(tif_files)}")
        print(f"  XML found: {len(xml_files)}")


        for tif in tif_files:

            smart_id = extract_planet_id(tif)

            records.append({

                "Project":
                    PROJECT_CONFIG["project_name"],

                "AOI":
                    aoi_name,

                "Smart_ID":
                    smart_id,

                "Date":
                    extract_datetime_from_id(smart_id),

                "Temporal_Mode":
                    PROJECT_CONFIG["temporal_mode"],

                "UAV_Date":
                    PROJECT_CONFIG["uav_date"],

                "TIF_Path":
                    tif,

                "XML_Path":
                    match_xml(
                        tif,
                        xml_files
                    ),

                "SHP_Path":
                    cfg["shp"]
            })



    df_inventory = pd.DataFrame(records)


    # ------------------------------------------------------------------------------
    # Temporal sorting
    # ------------------------------------------------------------------------------

    df_inventory = (
        df_inventory
        .sort_values(
            ["AOI", "Date"]
        )
        .reset_index(drop=True)
    )



    # ------------------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------------------

    print(
        """
    ==================================================
    PLANET INVENTORY CREATED
    ==================================================
    """
    )


    print("Total observations:", len(df_inventory))

    print(
        "\nObservations by AOI:"
    )

    display(
        df_inventory
        .groupby("AOI")
        ["Smart_ID"]
        .count()
    )


    print(
        "\nDate range:"
    )

    display(
        df_inventory
        .groupby("AOI")
        ["Date"]
        .agg(["min","max"])
    )


    print(
        "\nMissing XML:"
    )

    print(
        df_inventory["XML_Path"]
        .isna()
        .sum()
    )


    display(
        df_inventory.head()
    )

    # ===== [02_1 celda 2] =====
    # ==============================================================================
    # 3. PLANET METADATA EXTRACTION
    # ==============================================================================
    # Extract official PlanetScope metadata from XML files.
    #
    # Adds:
    # - acquisition geometry
    # - sensor information
    # - correction levels
    # - cloud / unusable data
    # ==============================================================================


    import xml.etree.ElementTree as ET
    import numpy as np



    def read_planet_xml(xml_path):
        """
        Extract Planet metadata fields from XML.
        """

        if xml_path is None:
            return {}


        tree = ET.parse(xml_path)
        root = tree.getroot()


        metadata = {}


        for elem in root.iter():

            tag = elem.tag.split("}")[-1]

            if elem.text is not None:

                value = elem.text.strip()

                if value != "":
                    metadata[tag] = value


        return metadata



    def safe_float(value):
        """
        Convert metadata values safely.
        """

        try:
            return float(value)

        except:
            return np.nan




    records = []


    for idx, row in df_inventory.iterrows():


        meta = read_planet_xml(
            row["XML_Path"]
        )


        records.append({

            # identifiers
            "AOI":
                row["AOI"],

            "Smart_ID":
                row["Smart_ID"],

            "Date":
                row["Date"],


            # SENSOR
            "Satellite":
                meta.get("serialIdentifier"),

            "Product_Type":
                meta.get("productType"),

            "Processing_Level":
                meta.get("geoCorrectionLevel"),


            # RESOLUTION
            "GSD_Row":
                safe_float(
                    meta.get("rowGsd")
                ),

            "GSD_Column":
                safe_float(
                    meta.get("columnGsd")
                ),


            # VIEW GEOMETRY
            "View_Angle":
                safe_float(
                    meta.get("spaceCraftViewAngle")
                ),

            "Incidence_Angle":
                safe_float(
                    meta.get("incidenceAngle")
                ),


            # SOLAR GEOMETRY
            "Solar_Azimuth":
                safe_float(
                    meta.get("illuminationAzimuthAngle")
                ),

            "Solar_Elevation":
                safe_float(
                    meta.get("illuminationElevationAngle")
                ),


            # QUALITY
            "Cloud_Percent":
                safe_float(
                    meta.get("cloudCoverPercentage")
                ),

            "Unusable_Percent":
                safe_float(
                    meta.get("unusableDataPercentage")
                ),


            # PROCESSING
            "DEM_Correction":
                meta.get(
                    "elevationCorrectionApplied"
                ),

            "Atmospheric_Correction":
                meta.get(
                    "atmosphericCorrectionApplied"
                )

        })



    df_metadata = pd.DataFrame(records)



    # Merge with inventory

    df_master = (
        df_inventory
        .merge(
            df_metadata,
            on=[
                "AOI",
                "Smart_ID",
                "Date"
            ],
            how="left"
        )
    )



    print(
    """
    ==================================================
    PLANET METADATA EXTRACTED
    ==================================================
    """
    )


    print("Total observations:", len(df_master))


    display(
        df_master[
            [
                "AOI",
                "Date",
                "Satellite",
                "View_Angle",
                "Cloud_Percent",
                "Unusable_Percent",
                "Processing_Level"
            ]
        ].head()
    )


    print("\nMissing metadata:")

    display(
        df_master.isna()
        .sum()
    )

    # ===== [02_1 celda 4] =====
    # ==============================================================================
    # 4. SOLAR AND TEMPORAL ACQUISITION GEOMETRY
    # ==============================================================================
    # Adds physically meaningful acquisition descriptors:
    #
    # - True Solar Time (NOAA equation of time)
    # - Solar Zenith Angle
    # - Hydrological year
    # - UAV temporal distance
    # ==============================================================================


    import numpy as np
    import pandas as pd



    # ------------------------------------------------------------------------------
    # NOAA TRUE SOLAR TIME
    # ------------------------------------------------------------------------------

    def calculate_true_solar_time(
        dates,
        longitude
    ):
        """
        NOAA Global Monitoring Division approximation.

        Corrects UTC acquisition time using:
        - Equation of Time
        - Longitude offset
        """

        gamma = (
            2
            * np.pi
            / 365
            *
            (
                dates.dt.dayofyear
                - 1
                +
                (dates.dt.hour - 12)
                / 24
            )
        )


        equation_time = (
            229.18
            *
            (
                0.000075
                +
                0.001868*np.cos(gamma)
                -
                0.032077*np.sin(gamma)
                -
                0.014615*np.cos(2*gamma)
                -
                0.040849*np.sin(2*gamma)
            )
        )


        time_offset = (
            equation_time
            +
            4 * longitude
        )


        minutes = (
            dates.dt.hour * 60
            +
            dates.dt.minute
            +
            dates.dt.second / 60
            +
            time_offset
        )


        return (
            minutes % 1440
        ) / 60



    # ------------------------------------------------------------------------------
    # Date formatting
    # ------------------------------------------------------------------------------

    df_master["Date"] = pd.to_datetime(
        df_master["Date"],
        utc=True
    )


    df_master["UAV_Date"] = pd.to_datetime(
        df_master["UAV_Date"],
        utc=True
    )



    # ------------------------------------------------------------------------------
    # Solar geometry
    # ------------------------------------------------------------------------------

    df_master["Solar_Zenith"] = (
        90
        -
        df_master["Solar_Elevation"]
    )



    # Need acquisition longitude
    # Extract approximate longitude from raster centroid

    import rasterio


    def get_raster_lon(path):

        try:

            with rasterio.open(path) as src:

                center_x = (
                    src.bounds.left
                    +
                    src.bounds.right
                ) / 2


                center_y = (
                    src.bounds.top
                    +
                    src.bounds.bottom
                ) / 2


                lon, lat = (
                    src.crs
                ), None


                from pyproj import Transformer

                transformer = Transformer.from_crs(
                    src.crs,
                    "EPSG:4326",
                    always_xy=True
                )


                lon, lat = transformer.transform(
                    center_x,
                    center_y
                )


                return lon

        except:

            return np.nan



    df_master["Longitude"] = (
        df_master["TIF_Path"]
        .apply(get_raster_lon)
    )



    df_master["True_Solar_Hour"] = (
        calculate_true_solar_time(
            df_master["Date"],
            df_master["Longitude"]
        )
    )



    df_master["Mean_Solar_Hour"] = (
        df_master["Date"].dt.hour
        +
        df_master["Date"].dt.minute/60
        +
        df_master["Longitude"]/15
    )



    # ------------------------------------------------------------------------------
    # Temporal descriptors
    # ------------------------------------------------------------------------------

    df_master["DOY"] = (
        df_master["Date"]
        .dt
        .dayofyear
    )


    df_master["Hydro_Year"] = (
        df_master["Date"]
        .apply(
            lambda x:
            x.year + 1
            if x.month >= 10
            else x.year
        )
    )


    df_master["Days_From_UAV"] = (
        df_master["Date"]
        -
        df_master["UAV_Date"]
    ).dt.days



    # ------------------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------------------

    print(
    """
    ==================================================
    SOLAR GEOMETRY COMPLETED
    ==================================================
    """
    )


    display(
        df_master[
            [
                "AOI",
                "Date",
                "True_Solar_Hour",
                "Solar_Zenith",
                "View_Angle",
                "Days_From_UAV"
            ]
        ]
        .head()
    )


    print(
        df_master[
            [
                "True_Solar_Hour",
                "Solar_Zenith",
                "View_Angle"
            ]
        ]
        .describe()
    )

    # ===== [02_1 celda 5] =====
    # ==============================================================================
    # 5. UAV → PLANET FIXED PIXEL GRID MAPPING
    # ==============================================================================
    # Creates or loads fixed Planet pixels derived from UAV polygons
    # ==============================================================================


    import rasterio
    import geopandas as gpd
    from rasterio.features import geometry_mask
    import pandas as pd
    import numpy as np
    import os


    PIXEL_DIR = os.path.join(
        PROJECT_CONFIG["output_dir"],
        "Pixel_Maps"
    )

    os.makedirs(
        PIXEL_DIR,
        exist_ok=True
    )


    pixel_maps = {}


    for aoi, cfg in PROJECT_CONFIG["aois"].items():


        print("\nProcessing:", aoi)


        outfile = os.path.join(
            PIXEL_DIR,
            f"{aoi}_pixels.parquet"
        )


        # ----------------------------------------------------------
        # Load cache
        # ----------------------------------------------------------

        if os.path.exists(outfile):

            pixels = pd.read_parquet(
                outfile
            )

            print(
                "[CACHE]",
                len(pixels),
                "pixels"
            )


        # ----------------------------------------------------------
        # Create pixel map
        # ----------------------------------------------------------

        else:


            ref_img = (
                df_master[
                    df_master["AOI"] == aoi
                ]
                ["TIF_Path"]
                .iloc[0]
            )


            gdf = gpd.read_file(
                cfg["shp"]
            )


            with rasterio.open(ref_img) as src:


                gdf = gdf.to_crs(
                    src.crs
                )


                mask = geometry_mask(

                    gdf.geometry,

                    transform=src.transform,

                    invert=True,

                    out_shape=(
                        src.height,
                        src.width
                    )
                )


                rows, cols = np.where(
                    mask
                )


                xs, ys = rasterio.transform.xy(

                    src.transform,

                    rows,

                    cols
                )



            pixels = pd.DataFrame({

                "pixel_id":
                    np.arange(len(rows)),

                "row":
                    rows,

                "col":
                    cols,

                "x":
                    xs,

                "y":
                    ys

            })



            pixels.to_parquet(

                outfile,

                index=False,

                compression="zstd"
            )


            print(
                "[CREATED]",
                len(pixels),
                "pixels"
            )



        pixel_maps[aoi] = pixels



    print(
        "\n================================"
    )

    print(
        "PIXEL MAPS READY"
    )

    for k,v in pixel_maps.items():

        print(
            k,
            "→",
            len(v),
            "pixels"
        )

    # ===== [02_1 celda 6] =====
    # ==============================================================================
    # 5B. UAV-DERIVED PIXEL VEGETATION PURITY
    # ==============================================================================
    # Objective:
    # Quantify vegetation fraction inside each Planet pixel
    # using UAV-derived polygons.
    #
    # This evaluates mixed pixel effects.
    #
    # Output:
    #   pixel_maps updated with Veg_Fraction
    # ==============================================================================


    import geopandas as gpd
    import pandas as pd
    import numpy as np
    import os

    from shapely.geometry import box



    # ------------------------------------------------------------------------------
    # Process each AOI
    # ------------------------------------------------------------------------------

    for aoi, pixels in pixel_maps.items():


        print(
            "\nProcessing:",
            aoi
        )


        cfg = PROJECT_CONFIG["aois"][aoi]


        # ----------------------------------------------------------
        # Load UAV polygons
        # ----------------------------------------------------------

        veg = gpd.read_file(
            cfg["shp"]
        )



        # ----------------------------------------------------------
        # Get Planet CRS and transform
        # ----------------------------------------------------------

        ref_img = (
            df_master[
                df_master["AOI"] == aoi
            ]
            ["TIF_Path"]
            .iloc[0]
        )


        with rasterio.open(ref_img) as src:


            veg = veg.to_crs(
                src.crs
            )


            transform = src.transform


            pixel_size = src.res[0]



        # ----------------------------------------------------------
        # Create Planet pixel polygons
        # ----------------------------------------------------------

        pixel_polygons = []


        for _, p in pixels.iterrows():


            x = p["x"]
            y = p["y"]


            poly = box(

                x - pixel_size/2,
                y - pixel_size/2,

                x + pixel_size/2,
                y + pixel_size/2

            )


            pixel_polygons.append(
                poly
            )



        gdf_pixels = gpd.GeoDataFrame(

            pixels.copy(),

            geometry=pixel_polygons,

            crs=veg.crs

        )



        # ----------------------------------------------------------
        # Intersection UAV vegetation vs Planet pixels
        # ----------------------------------------------------------

        inter = gpd.overlay(

            gdf_pixels,

            veg,

            how="intersection"

        )



        inter["veg_area"] = (
            inter.geometry.area
        )



        veg_fraction = (

            inter

            .groupby("pixel_id")

            ["veg_area"]

            .sum()

            /

            (pixel_size ** 2)

        )



        gdf_pixels["Veg_Fraction"] = (

            gdf_pixels["pixel_id"]

            .map(
                veg_fraction
            )

            .fillna(0)

        )



        # ----------------------------------------------------------
        # Remove geometry and update
        # ----------------------------------------------------------

        pixel_maps[aoi] = (

            pd.DataFrame(
                gdf_pixels.drop(
                    columns="geometry"
                )
            )

        )



        print(
            "Pixels:",
            len(pixel_maps[aoi])
        )


        print(
            "Median vegetation fraction:",
            round(
                pixel_maps[aoi]["Veg_Fraction"].median(),
                3
            )
        )


        print(
            "Pixels >80% pure:",
            (
                pixel_maps[aoi]["Veg_Fraction"]
                >=0.8
            )
            .sum()
        )

    # ===== [02_1 celda 7] =====
    # ==============================================================================
    # 6. FIXED PIXEL AVAILABILITY QUALITY CONTROL
    # ==============================================================================
    # Objective:
    # Check whether UAV-derived Planet pixels contain valid data
    # in each Planet acquisition.
    #
    # Replaces old AOI_Coverage raster.mask approach.
    # ==============================================================================


    import rasterio
    import pandas as pd
    import numpy as np
    import os
    from tqdm import tqdm



    QC_FILE = os.path.join(
        PROJECT_CONFIG["output_dir"],
        "Pixel_Availability_QC.parquet"
    )



    # ------------------------------------------------------------------------------
    # Load cache
    # ------------------------------------------------------------------------------

    if os.path.exists(QC_FILE):

        print("[CACHE] Loading pixel availability")

        df_pixel_qc = pd.read_parquet(
            QC_FILE
        )


    else:


        print("[RUN] Computing pixel availability")


        records = []


        for _, row in tqdm(
            df_master.iterrows(),
            total=len(df_master)
        ):


            pixels = pixel_maps[
                row["AOI"]
            ]


            rows = pixels["row"].values
            cols = pixels["col"].values


            try:


                with rasterio.open(
                    row["TIF_Path"]
                ) as src:


                    # NIR band
                    nir = src.read(
                        8
                    )


                    values = nir[
                        rows,
                        cols
                    ]


                coverage = (
                    np.sum(values > 0)
                    /
                    len(values)
                )


            except Exception:


                coverage = 0



            records.append({

                "AOI":
                    row["AOI"],

                "Smart_ID":
                    row["Smart_ID"],

                "Pixel_Coverage":
                    coverage

            })



        df_pixel_qc = pd.DataFrame(
            records
        )



        df_pixel_qc.to_parquet(

            QC_FILE,

            index=False,

            compression="zstd"

        )


        print("[CACHE SAVED]")



    # ------------------------------------------------------------------------------
    # Merge into master table
    # ------------------------------------------------------------------------------

    df_master = df_master.merge(

        df_pixel_qc,

        on=[
            "AOI",
            "Smart_ID"
        ],

        how="left"
    )



    # ------------------------------------------------------------------------------
    # QC flag
    # ------------------------------------------------------------------------------

    PIXEL_COVERAGE_THRESHOLD = 0.99


    df_master["QC_Pixel_Coverage"] = (

        df_master["Pixel_Coverage"]

        >=

        PIXEL_COVERAGE_THRESHOLD

    )



    # ------------------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------------------

    print(
    """
    ==================================================
    PIXEL AVAILABILITY QC
    ==================================================
    """
    )


    display(

        df_master

        .groupby("AOI")

        .agg(

            Total=("Smart_ID","count"),

            Accepted=("QC_Pixel_Coverage","sum"),

            Median_Coverage=("Pixel_Coverage","median"),

            Min_Coverage=("Pixel_Coverage","min")

        )

    )

    # ===== [02_1 celda 11] =====
    import rasterio
    import pandas as pd
    import numpy as np
    import os
    from tqdm import tqdm



    # ==============================================================================
    # OUTPUT CACHE
    # ==============================================================================

    PIXEL_SPECTRAL_FILE = os.path.join(

        PROJECT_CONFIG["output_dir"],

        "Pixel_Spectral_Master_v2.parquet"

    )



    # ==============================================================================
    # PLANETSCOPE SUPERDOVE BAND DEFINITION
    # ==============================================================================

    BANDS = {

        "Coastal_Blue": 1,

        "Blue": 2,

        "Green_I": 3,

        "Green": 4,

        "Yellow": 5,

        "Red": 6,

        "RedEdge": 7,

        "NIR": 8

    }



    # ==============================================================================
    # EXTRACTION
    # ==============================================================================

    if os.path.exists(
        PIXEL_SPECTRAL_FILE
    ):


        print(
            "[CACHE] Loading pixel spectral database"
        )


        df_pixels = pd.read_parquet(

            PIXEL_SPECTRAL_FILE

        )



    else:


        print(
            "[RUN] Extracting Planet fixed pixels"
        )


        records = []



        df_input = (

            df_master[

                df_master["QC_Pixel_Coverage"]

            ]

            .copy()

        )



        for _, row in tqdm(

            df_input.iterrows(),

            total=len(df_input)

        ):



            # ------------------------------------------------------
            # Fixed UAV-derived pixels
            # ------------------------------------------------------

            pixels = pixel_maps[

                row["AOI"]

            ]



            rows = pixels["row"].values

            cols = pixels["col"].values



            # ------------------------------------------------------
            # Read Planet image
            # ------------------------------------------------------

            with rasterio.open(

                row["TIF_Path"]

            ) as src:



                img = (

                    src.read()

                    .astype(float)

                    /

                    10000

                )



                values = img[

                    :,

                    rows,

                    cols

                ]



            # ------------------------------------------------------
            # Base dataframe
            # ------------------------------------------------------

            temp = pd.DataFrame({


                "AOI":

                    row["AOI"],


                "Smart_ID":

                    row["Smart_ID"],


                "Date":

                    row["Date"],


                "pixel_id":

                    pixels["pixel_id"].values,


                "x":

                    pixels["x"].values,


                "y":

                    pixels["y"].values,


                "Veg_Fraction":

                    pixels["Veg_Fraction"].values,


                "True_Solar_Hour":

                    row["True_Solar_Hour"],


                "Solar_Zenith":

                    row["Solar_Zenith"],


                "View_Angle":

                    row["View_Angle"]

            })



            # ------------------------------------------------------
            # Add spectral bands
            # ------------------------------------------------------

            for name, idx in BANDS.items():


                temp[name] = values[

                    idx-1

                ]



            records.append(

                temp

            )




        # ==========================================================================
        # CONCATENATE
        # ==========================================================================


        df_pixels = pd.concat(

            records,

            ignore_index=True

        )



        # ==========================================================================
        # VEGETATION INDICES
        # ==========================================================================


        EPS = 1e-10



        def norm_index(a,b):

            return (

                (a-b)

                /

                (a+b+EPS)

            )



        # --------------------------------------------------------------------------
        # Classical indices
        # --------------------------------------------------------------------------

        df_pixels["NDVI"] = norm_index(

            df_pixels["NIR"],

            df_pixels["Red"]

        )



        df_pixels["NDRE"] = norm_index(

            df_pixels["NIR"],

            df_pixels["RedEdge"]

        )



        df_pixels["GNDVI"] = norm_index(

            df_pixels["NIR"],

            df_pixels["Green"]

        )



        # --------------------------------------------------------------------------
        # Planet PRI proxy
        # --------------------------------------------------------------------------
        # Original PRI:
        # (R531 - R570)/(R531 + R570)
        #
        # Planet approximation:
        # Green_I - Green_I
        # --------------------------------------------------------------------------


        df_pixels["PRI_proxy"] = norm_index(

            df_pixels["Green"],

            df_pixels["Green_I"]

        )



        # --------------------------------------------------------------------------
        # RedEdge related metrics
        # --------------------------------------------------------------------------

        df_pixels["RE_NIR_ratio"] = (

            df_pixels["RedEdge"]

            /

            (

                df_pixels["NIR"]

                +

                EPS

            )

        )



        df_pixels["RE_contrast"] = norm_index(

            df_pixels["RedEdge"],

            df_pixels["Red"]

        )



        # ==========================================================================
        # SAVE CACHE
        # ==========================================================================


        df_pixels.to_parquet(

            PIXEL_SPECTRAL_FILE,

            index=False,

            compression="zstd"

        )



        print(

            "[CACHE SAVED]"

        )



    # ==============================================================================
    # SUMMARY
    # ==============================================================================

    print(
    """
    ==================================================
    PIXEL SPECTRAL DATABASE READY
    ==================================================
    """
    )


    print(

        "Rows:",

        len(df_pixels)

    )



    display(

        df_pixels

        .groupby(

            "AOI"

        )

        .agg(

            Pixels=(

                "pixel_id",

                "nunique"

            ),

            Images=(

                "Smart_ID",

                "nunique"

            ),

            NDVI_mean=(

                "NDVI",

                "mean"

            ),

            NDRE_mean=(

                "NDRE",

                "mean"

            ),

            GNDVI_mean=(

                "GNDVI",

                "mean"

            ),

            PRI_mean=(

                "PRI_proxy",

                "mean"

            ),

            Mean_VegFraction=(

                "Veg_Fraction",

                "mean"

            )

        )

    )

### 0.3 Ejecutar construcción para todos los proyectos (no-op si la caché existe)

In [ ]:
# Section 0 — driver: construye (si faltan) los parquets de CADA proyecto.
# Usa la config de proyectos heredada de 02_1 (rutas a TIFs/shapefiles).
for _key, _cfg in PROJECTS.items():
    build_project_caches(_cfg)
print('\nSection 0 lista: parquets disponibles para la carga de abajo.')


In [ ]:
# ------------------------------------------------------------------------------
# 0.4 GENERACIÓN DE METADATA_MASTER_NOAA.CSV POR PROYECTO
# ------------------------------------------------------------------------------
import os
import glob
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from shapely.geometry import Polygon
from sklearn.cluster import KMeans
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, *args, **kwargs): return x

# ------------------------------------------------------------------------------
# A. FUNCIÓN DE FÍSICA SOLAR
# ------------------------------------------------------------------------------
def calculate_true_solar_time_vectorized(dates, lons):
    gamma = 2 * np.pi / 365.0 * (dates.dt.dayofyear - 1 + (dates.dt.hour - 12) / 24.0)
    eqtime = 229.18 * (0.000075 + 0.001868 * np.cos(gamma) - 0.032077 * np.sin(gamma) \
             - 0.014615 * np.cos(2 * gamma) - 0.040849 * np.sin(2 * gamma))
    time_offset = eqtime + 4 * lons
    tst_minutes = dates.dt.hour * 60 + dates.dt.minute + dates.dt.second / 60 + time_offset
    return (tst_minutes % 1440) / 60.0

# ------------------------------------------------------------------------------
# B. PARSEO DE XML 
# ------------------------------------------------------------------------------
NS = {
    'gml': 'http://www.opengis.net/gml',
    'ps': 'http://schemas.planet.com/ps/v1/planet_product_metadata_geocorrected_level',
    'eop': 'http://earth.esa.int/eop',
    'opt': 'http://earth.esa.int/opt'
}

def get_smart_id(filename):
    parts = filename.split('_')
    if len(parts) >= 4:
        return "_".join(parts[:4])
    return os.path.splitext(filename)[0]

def parse_planet_xml_raw(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        filename = os.path.basename(xml_path)
        smart_id = get_smart_id(filename)
        
        date_str = root.find('.//eop:acquisitionDate', NS).text
        acq_dt = pd.to_datetime(date_str) 
        
        coords_node = root.find('.//gml:coordinates', NS)
        centroid_x, centroid_y = np.nan, np.nan
        if coords_node is not None:
            pts = [tuple(map(float, c.split(','))) for c in coords_node.text.strip().split()]
            if len(pts) > 2:
                poly = Polygon(pts)
                centroid_x = poly.centroid.x
                centroid_y = poly.centroid.y
                
        inc_angle = float(root.find('.//eop:incidenceAngle', NS).text)
        elev_angle = float(root.find('.//opt:illuminationElevationAngle', NS).text)
        cloud_cov = float(root.find('.//opt:cloudCoverPercentage', NS).text)
        unusable = float(root.find('.//ps:unusableDataPercentage', NS).text)
        
        return {
            "Smart_ID": smart_id,
            "File": filename,
            "Date": acq_dt,
            "Lon": centroid_x,
            "Lat": centroid_y,
            "Sensor_Incidence": inc_angle,
            "Solar_Elevation": elev_angle,
            "Solar_Zenith": 90 - elev_angle,
            "Cloud_Cover": cloud_cov,
            "Unusable_Pct": unusable,
            "XML_Path": xml_path
        }
    except Exception:
        return None

# ------------------------------------------------------------------------------
# C. EJECUCIÓN DEL FLUJO DE TRABAJO (Adaptado a diccionarios locales)
# ------------------------------------------------------------------------------
for project_name, config in PROJECTS.items():
    print(f"\n==================================================")
    print(f"GENERANDO METADATA PARA: {project_name}")
    print(f"==================================================")
    
    out_path = os.path.join(config['output_dir'], "Metadata_Master_NOAA.csv")
    
    if os.path.exists(out_path):
        print(f"[CACHE OK] El CSV de metadatos ya existe: {out_path}")
        continue

    # Extraer carpetas de imágenes del proyecto activo
    target_dirs = [aoi_conf['img_dir'] for aoi_conf in config['aois'].values()]
    all_xmls = []
    for d in target_dirs:
        all_xmls.extend(glob.glob(os.path.join(d, "**", "*.xml"), recursive=True))
    all_xmls = list(set(all_xmls))

    if not all_xmls:
        print(f"[ERROR] No se encontraron XMLs para {project_name}.")
        continue

    print(f">>> Parseando {len(all_xmls)} archivos XML crudos...")
    records = []
    for xml in tqdm(all_xmls):
        data = parse_planet_xml_raw(xml)
        if data:
            records.append(data)

    df_meta = pd.DataFrame(records)

    if not df_meta.empty:
        print(">>> Calculando True Solar Time (NOAA EOT)...")
        df_meta['True_Solar_Hour'] = calculate_true_solar_time_vectorized(df_meta['Date'], df_meta['Lon'])
        df_meta['Mean_Solar_Hour'] = df_meta['Date'].dt.hour + df_meta['Date'].dt.minute/60.0 + df_meta['Lon']/15.0

        print(">>> Calculando Jitter Geométrico...")
        coords = df_meta[['Lon', 'Lat']].dropna().values
        if len(coords) > 0:
            n_clusters = 2 if len(coords) > 10 else 1
            kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(coords)
            df_meta['AOI_Cluster'] = kmeans.labels_
            
            df_meta['Jitter_m'] = np.nan
            deg_lat_m = 110950
            deg_lon_m = 88900 
            
            for cluster in df_meta['AOI_Cluster'].unique():
                mask = df_meta['AOI_Cluster'] == cluster
                mean_lon = df_meta.loc[mask, 'Lon'].mean()
                mean_lat = df_meta.loc[mask, 'Lat'].mean()
                
                d_lon = (df_meta.loc[mask, 'Lon'] - mean_lon) * deg_lon_m
                d_lat = (df_meta.loc[mask, 'Lat'] - mean_lat) * deg_lat_m
                df_meta.loc[mask, 'Jitter_m'] = np.sqrt(d_lon**2 + d_lat**2)

        df_meta.to_csv(out_path, index=False)
        print(f"[SUCCESS] Guardado en: {out_path}")
        print(f"    - Imágenes totales: {len(df_meta)}")
        print(f"    - Jitter Medio:     {df_meta['Jitter_m'].mean():.3f} m")
    else:
        print(f"[ERROR] DataFrame vacío. No se generaron registros para {project_name}.")

---
## Sección 0-bis — Carga de los parquets y registro de proyectos (original de 02_2)
_A partir de aquí, todo es el notebook 02_2 original sin cambios._

# PlanetScope SuperDove — Data Quality Diagnostics for Paper
## Cross-Project Comparison: *Corema album* vs *Avocado/Mango*

**Purpose:** Reproduce the figures and tables reported in §3.1 of the manuscript for both study sites simultaneously, comparing their geometric stability, acquisition geometry, orbital drift and intra-day radiometric consistency.

**Data source:** Pre-computed Parquet files from `02_3_audit_PlanetScope.ipynb` (no raw image processing required here).

| Section | Content | Paper reference |
|---------|---------|----------------|
| 0 | Configuration & data load | — |
| 1 | Dataset summary (Table 2) | §3.1 |
| 2 | View angle distribution | Fig. 4 |
| 3 | Orbital drift & TST | Fig. 5 |
| 4 | Intra-day spectral concordance (Lin's CCC) | Fig. 6 |
| 5 | NDVI drift propagation | §3.1.4 |
| 6 | Cross-project statistical comparison | §3.1 |

---


## 0  Configuration & Data Load

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, ks_2samp
warnings.filterwarnings("ignore")

# ─── Project registry ─────────────────────────────────────────────────────────
BASE = r".\03_figures_&_results"

PROJECTS = {
    "Corema album": {
        "pixel_file": os.path.join(BASE, "01_corema_album", "Pixel_Spectral_Master_v2.parquet"),
        "qc_file":    os.path.join(BASE, "01_corema_album", "Pixel_Availability_QC.parquet"),
        "output_dir": os.path.join(BASE, "01_corema_album"),
        "color":      "#2874A6",
        "color_light":"#85C1E9",
        "marker":     "o",
    },
    "Avocado/Mango": {
        "pixel_file": os.path.join(BASE, "02_avocado_mango", "Pixel_Spectral_Master_v2.parquet"),
        "qc_file":    os.path.join(BASE, "02_avocado_mango", "Pixel_Availability_QC.parquet"),
        "output_dir": os.path.join(BASE, "02_avocado_mango"),
        "color":      "#C0392B",
        "color_light":"#F1948A",
        "marker":     "s",
    },
}

OUTPUT_DIR = os.path.join(BASE, "00_paper_comparison")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── Global style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":     "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size":        12,
    "axes.labelsize":   13,
    "axes.titlesize":   13,
    "xtick.labelsize":  11,
    "ytick.labelsize":  11,
    "legend.fontsize":  10,
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "grid.linestyle":   "--",
    "figure.dpi":       100,
})

# ─── Load ─────────────────────────────────────────────────────────────────────
frames, qc_frames = [], []

for name, cfg in PROJECTS.items():
    df = pd.read_parquet(cfg["pixel_file"])
    df["Project"] = name
    df["Date"] = pd.to_datetime(df["Date"], utc=True)
    df["Date_only"] = df["Date"].dt.date
    frames.append(df)

    qc = pd.read_parquet(cfg["qc_file"])
    qc["Project"] = name
    qc_frames.append(qc)

df_pixels = pd.concat(frames, ignore_index=True)
df_qc_all = pd.concat(qc_frames, ignore_index=True)

# One row per image (unique Smart_ID x AOI x Project)
df_images = (
    df_pixels
    .sort_values(["Project", "AOI", "Date"])
    .drop_duplicates(subset=["Project", "AOI", "Smart_ID"])
    [["Project","AOI","Smart_ID","Date","Date_only",
      "True_Solar_Hour","Solar_Zenith","View_Angle"]]
    .copy()
)

print("=" * 62)
print("DATASETS LOADED")
print("=" * 62)
for name, g in df_images.groupby("Project"):
    print(f"\n  {name}")
    print(f"    Images:     {len(g):,}")
    print(f"    AOIs:       {sorted(g['AOI'].unique())}")
    print(f"    Date range: {g['Date'].min().date()} -> {g['Date'].max().date()}")
print()


# ─── Entity registry (Avocado/Mango split from Section 4 onwards) ────────────
def assign_entity(row):
    if row["Project"] == "Corema album":
        return "Corema album"
    if row["AOI"] == "avocado":
        return "Avocado"
    if row["AOI"] == "mango":
        return "Mango"
    return row["Project"]   # fallback

df_pixels["Entity"] = df_pixels.apply(assign_entity, axis=1)
df_images["Entity"] = df_images.apply(assign_entity, axis=1)

ENTITIES = {
    "Corema album": {"color": "#2874A6", "color_light": "#85C1E9", "marker": "o"},
    "Avocado":      {"color": "#27AE60", "color_light": "#82E0AA", "marker": "^"},
    "Mango":        {"color": "#E67E22", "color_light": "#F0B27A", "marker": "s"},
}

print("\nEntities (from Section 4 onwards):")
for ename, grp in df_pixels.groupby("Entity"):
    n_img = grp["Smart_ID"].nunique()
    n_pix = int(grp["pixel_id"].nunique())
    ndvi  = grp["NDVI"].median()
    vf    = grp["Veg_Fraction"].median() if "Veg_Fraction" in grp.columns else float("nan")
    print(f"  {ename:<16}  images={n_img:>5}  pixels={n_pix:>5}  "
          f"med_NDVI={ndvi:.3f}  med_VF={vf:.3f}")


---
## 1  Dataset Summary  (Table 2)

For each project: total image stack, spatially valid images, rejection rate,
valid intra-day pairs (\u0394t \u2265 40 min), and independent calendar dates for radiometric validation.


In [ ]:
GAP_MIN = 40
rows = []

for name, cfg in PROJECTS.items():

    imgs = df_images[df_images["Project"] == name]
    qc   = df_qc_all[df_qc_all["Project"] == name]

    total_stack  = len(qc)
    valid_images = len(imgs)
    rej_pct      = (1 - valid_images / total_stack) * 100

    daily = (
        imgs.groupby(["AOI", "Date_only"])
        .agg(
            TST_min=("True_Solar_Hour", "min"),
            TST_max=("True_Solar_Hour", "max"),
        )
        .assign(gap_min=lambda d: (d["TST_max"] - d["TST_min"]) * 60)
        .reset_index()
    )
    revisits = daily[daily["gap_min"] >= GAP_MIN]

    rows.append({
        "Project":                              name,
        "Total image stack (n)":               total_stack,
        "Valid images (pixel coverage >= 99%)": valid_images,
        "Rejection rate (%)":                  round(rej_pct, 2),
        "Valid intra-day pairs (dt >= 40 min)": len(revisits),
        "Calendar dates for validation":        int(revisits["Date_only"].nunique()),
    })

df_table2 = pd.DataFrame(rows).set_index("Project").T

print("=" * 70)
print("TABLE 2 - DATA QUALITY WORKFLOW SUMMARY")
print("=" * 70)
display(df_table2)

out = os.path.join(OUTPUT_DIR, "Table2_DataQuality_Summary.csv")
df_table2.to_csv(out)
print(f"\n[SAVED] {out}")


---
## 2  Sensor Viewing Geometry  (Figure 4)

Distribution of sensor off-nadir angles for both projects.
Vertical lines: nadir (0 deg), stability envelope (5 deg), maximum accepted (8 deg).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

for ax, (name, cfg) in zip(axes, PROJECTS.items()):

    x  = df_images[df_images["Project"] == name]["View_Angle"].dropna()
    ax2 = ax.twinx()

    sns.histplot(x, bins=40, stat="count",
                 color=cfg["color"], alpha=0.35,
                 edgecolor="white", linewidth=0.4,
                 label="Count", ax=ax)
    sns.kdeplot(x, color=cfg["color"], linewidth=2.5,
                fill=False, ax=ax2)

    ax2.set_ylabel("Probability density", color=cfg["color"], fontsize=11)
    ax2.tick_params(axis="y", labelcolor=cfg["color"])
    ax2.grid(False)

    ax.axvline( 0, color="black",   linestyle="-",  linewidth=1.2, alpha=0.45, label="Nadir")
    ax.axvline( 5, color="#C0392B", linestyle="--", linewidth=1.8, label="+/-5 deg")
    ax.axvline(-5, color="#C0392B", linestyle="--", linewidth=1.8)
    ax.axvline( 8, color="#E67E22", linestyle=":",  linewidth=1.5, label="+/-8 deg")
    ax.axvline(-8, color="#E67E22", linestyle=":",  linewidth=1.5)

    stats_text = (
        f"n = {len(x):,}\n"
        f"Mean:    {x.mean():+.2f} deg\n"
        f"sigma:   {x.std():.2f} deg\n"
        f"|<=5 deg|: {(x.abs()<=5).mean()*100:.1f}%\n"
        f"|<=8 deg|: {(x.abs()<=8).mean()*100:.1f}%"
    )
    ax.text(0.03, 0.97, stats_text, transform=ax.transAxes,
            fontsize=10, va="top", family="monospace",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                      alpha=0.92, edgecolor="#bbb"))

    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Sensor off-nadir angle (degrees)")
    ax.set_ylabel("Number of acquisitions")
    ax.legend(loc="upper right", fontsize=9, framealpha=0.9)

# plt.suptitle("Figure 4 - Distribution of Sensor Viewing Geometry",
#              fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()

out4 = os.path.join(OUTPUT_DIR, "Fig4_ViewAngle_Comparison.png")
plt.savefig(out4, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out4}")


---
## 3  Acquisition Timing and Orbital Drift  (Figure 5)

Temporal evolution of Local True Solar Time colour-coded by Solar Zenith Angle.
The dashed line is the 60-day centred rolling mean (orbital drift trend).


In [ ]:
# fig, axes = plt.subplots(2, 1, figsize=(14, 11), sharex=False)

# for ax, (name, cfg) in zip(axes, PROJECTS.items()):

#     d = (
#         df_images[df_images["Project"] == name]
#         .sort_values("Date").copy()
#     )
#     d["Rolling_TST"] = (
#         d["True_Solar_Hour"].rolling(window=60, center=True).mean()
#     )

#     sc = ax.scatter(
#         d["Date"], d["True_Solar_Hour"],
#         c=d["Solar_Zenith"], cmap="plasma",
#         s=25, alpha=0.7, edgecolor="none",
#         label="Acquisition (colour = SZA)"
#     )
#     ax.plot(d["Date"], d["Rolling_TST"],
#             color="white", linestyle="--", linewidth=3, alpha=0.8)
#     ax.plot(d["Date"], d["Rolling_TST"],
#             color="black", linestyle="--", linewidth=1.5,
#             label="60-day orbital drift")

#     cbar = plt.colorbar(sc, ax=ax, pad=0.01, shrink=0.88)
#     cbar.set_label("Solar Zenith Angle (deg)", fontsize=10)
#     cbar.outline.set_linewidth(0)

#     # Annotate drift amplitude (first vs last 25% of series)
#     t25 = d["Date"].quantile(0.25)
#     t75 = d["Date"].quantile(0.75)
#     early_tst = d[d["Date"] <= t25]["True_Solar_Hour"].mean()
#     late_tst  = d[d["Date"] >= t75]["True_Solar_Hour"].mean()
#     drift_min = (late_tst - early_tst) * 60
#     ax.text(0.99, 0.04,
#             f"Drift: {drift_min:+.0f} min (early -> late)",
#             transform=ax.transAxes, ha="right", fontsize=10,
#             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
#                       alpha=0.88, edgecolor="#aaa"))

#     ax.set_ylabel("Local True Solar Time (h)", fontweight="bold")
#     ax.set_title(name, fontweight="bold")
#     ax.xaxis.set_major_locator(mdates.YearLocator())
#     ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
#     ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
#     ax.legend(loc="upper left", fontsize=10,
#               facecolor="white", framealpha=0.9)
#     ax.grid(True, linestyle=":", alpha=0.5)

# axes[-1].set_xlabel("Date", fontweight="bold")
# plt.suptitle("Figure 5 - Orbital Drift and Solar Zenith Angle Evolution",
#              fontsize=14, fontweight="bold")
# plt.tight_layout()

# out5 = os.path.join(OUTPUT_DIR, "Fig5_OrbitalDrift_Comparison.png")
# plt.savefig(out5, dpi=300, bbox_inches="tight")
# plt.show()
# print(f"[SAVED] {out5}")

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import os

# Se asume que df_images, PROJECTS y OUTPUT_DIR ya están definidos en el entorno.

fig, axes = plt.subplots(2, 1, figsize=(14, 11), sharex=False)
GAP_HORAS = 40 / 60.0

for ax, (name, cfg) in zip(axes, PROJECTS.items()):

    # 1. Preparación del subconjunto por proyecto
    d = (
        df_images[df_images["Project"] == name]
        .sort_values("Date").copy()
    )
    d["Rolling_TST"] = (
        d["True_Solar_Hour"].rolling(window=60, center=True).mean()
    )

    # 2. Detección de Gaps (>40 min) específicos para este proyecto
    d["Date_Only"] = d["Date"].dt.date
    # transform(np.ptp) calcula el rango (max - min) por día y lo asigna a cada fila
    daily_range = d.groupby("Date_Only")["True_Solar_Hour"].transform(np.ptp)
    subset_gap = d[daily_range > GAP_HORAS]
    n_gap_days = subset_gap["Date_Only"].nunique()

    # 3. Ploteo: Capa A - Puntos de Adquisición (SZA)
    sc = ax.scatter(
        d["Date"], d["True_Solar_Hour"],
        c=d["Solar_Zenith"], cmap="plasma",
        s=25, alpha=0.7, edgecolor="none",
        label="Acquisition (colour = SZA)"
    )

    # 3. Ploteo: Capa B - Alerta de Gaps
    if n_gap_days > 0:
        ax.scatter(
            subset_gap["Date"], subset_gap["True_Solar_Hour"],
            facecolors="none", edgecolors="red",
            s=80, linewidth=1.5,
            label=f"Revisit >40min (n={n_gap_days} days)"
        )

    # 3. Ploteo: Capa C - Tendencia Orbital (Drift)
    ax.plot(d["Date"], d["Rolling_TST"],
            color="white", linestyle="--", linewidth=3, alpha=0.8)
    ax.plot(d["Date"], d["Rolling_TST"],
            color="black", linestyle="--", linewidth=1.5,
            label="60-day orbital drift")

    # 4. Formato de Colorbar
    cbar = plt.colorbar(sc, ax=ax, pad=0.01, shrink=0.88)
    cbar.set_label("Solar Zenith Angle (deg)", fontsize=10)
    cbar.outline.set_linewidth(0)

    # 5. Anotación de amplitud de drift (primer 25% vs último 25%)
    t25 = d["Date"].quantile(0.25)
    t75 = d["Date"].quantile(0.75)
    early_tst = d[d["Date"] <= t25]["True_Solar_Hour"].mean()
    late_tst  = d[d["Date"] >= t75]["True_Solar_Hour"].mean()
    drift_min = (late_tst - early_tst) * 60
    ax.text(0.99, 0.04,
            f"Drift: {drift_min:+.0f} min (early -> late)",
            transform=ax.transAxes, ha="right", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      alpha=0.88, edgecolor="#aaa"))

    # 6. Configuración de Ejes y Leyenda
    ax.set_ylabel("Local True Solar Time (h)", fontweight="bold")
    ax.set_title(name, fontweight="bold")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
    
    # Leyenda: Se agrupan los labels dinámicamente
    ax.legend(loc="upper left", fontsize=10,
              facecolor="white", framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.5)

axes[-1].set_xlabel("Date", fontweight="bold")
# plt.suptitle("Figure 5 - Orbital Drift, SZA and Revisit Consistency",
#              fontsize=14, fontweight="bold")
plt.tight_layout()

out5 = os.path.join(OUTPUT_DIR, "Fig5_OrbitalDrift_Comparison.png")
plt.savefig(out5, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out5}")


---
## 4  Intra-day Radiometric Stability (Figure 6)

For each valid same-day revisit pair (same AOI, same date, \u0394t \u2265 40 min)
fixed pixels are matched between early and late acquisitions.

**Primary metrics (cross-entity comparable):** RMSE and relative bias.
**Secondary (table only):** Lin's CCC — included for completeness but interpret
with caution: CCC is depressed by low spatial variance in radiometrically
homogeneous canopies (avocado), not by poor radiometric consistency. This is a
well-known dynamic-range compression artefact and not an indicator of unreliability.
The relative RMSE (RMSE normalised by mean reflectance) confirms that per-unit
uncertainty is comparable across all three entities.

> **From this section onwards Avocado and Mango are separate entities.**


In [ ]:
PURITY_OFF = True          # True = sin filtro de fc (todos los píxeles)
VF_MIN_GLOBAL = 0.0 if PURITY_OFF else 0.70

In [ ]:
BAND_COLS = [
    ("Coastal_Blue", "Coastal"),
    ("Blue",         "Blue"),
    ("Green_I",      "Green I"),
    ("Green",        "Green"),
    ("Yellow",       "Yellow"),
    ("Red",          "Red"),
    ("RedEdge",      "RedEdge"),
    ("NIR",          "NIR"),
]

def lin_ccc(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mu_t, mu_p   = y_true.mean(), y_pred.mean()
    var_t, var_p = y_true.var(), y_pred.var()
    sd_t, sd_p   = np.std(y_true), np.std(y_pred)
    r = np.corrcoef(y_true, y_pred)[0, 1]
    denom = var_t + var_p + (mu_t - mu_p)**2
    return (2 * r * sd_t * sd_p) / denom if denom > 0 else np.nan


def build_pairs(df_entity):
    early_store = {col: [] for col, _ in BAND_COLS}
    late_store  = {col: [] for col, _ in BAND_COLS}
    n_pairs = 0

    daily = (
        df_entity.groupby(["AOI", "Date_only", "Smart_ID"])["True_Solar_Hour"]
        .first().reset_index()
        .groupby(["AOI", "Date_only"])
        .agg(tst_min=("True_Solar_Hour", "min"),
             tst_max=("True_Solar_Hour", "max"))
        .reset_index()
    )
    daily["gap_min"] = (daily["tst_max"] - daily["tst_min"]) * 60
    valid_days = daily[daily["gap_min"] >= 40]

    for _, row in valid_days.iterrows():
        day = df_entity[
            (df_entity["AOI"] == row["AOI"]) &
            (df_entity["Date_only"] == row["Date_only"])
        ]
        img_t = (
            day.groupby("Smart_ID")["True_Solar_Hour"]
            .first().sort_values()
        )
        if len(img_t) < 2:
            continue
        e = day[day["Smart_ID"] == img_t.index[0]].set_index("pixel_id")
        l = day[day["Smart_ID"] == img_t.index[-1]].set_index("pixel_id")
        common = e.index.intersection(l.index)
        if len(common) < 5:
            continue
        for col, _ in BAND_COLS:
            if col in e.columns:
                early_store[col].extend(e.loc[common, col].values.tolist())
                late_store[col].extend(l.loc[common, col].values.tolist())
        n_pairs += 1

    return early_store, late_store, n_pairs


# ── Build pairs per entity ─────────────────────────────────────────────────────
ccc_results = []
pair_stores  = {}

print("Intra-day pairs per entity:")
for ename, grp in df_pixels.groupby("Entity"):
    early, late, n_pairs = build_pairs(grp)
    pair_stores[ename] = (early, late, n_pairs)
    print(f"  {ename}: {n_pairs} pairs")

    for col, label in BAND_COLS:
        x = np.array(early[col]);  y = np.array(late[col])
        mask = np.isfinite(x) & np.isfinite(y);  x, y = x[mask], y[mask]
        if len(x) < 10:
            continue
        mean_x   = float(np.mean(x))
        rmse     = float(np.sqrt(np.mean((y-x)**2)))
        bias_abs = float(np.mean(y-x))
        bias_rel = bias_abs / mean_x * 100
        ccc_results.append({
            "Entity":    ename,
            "Band":      label,
            "Column":    col,
            "CCC":       lin_ccc(x, y),
            "RMSE":      rmse,
            "rel_RMSE":  rmse / mean_x * 100,   # normalised, cross-entity comparable
            "Bias_abs":  bias_abs,
            "Bias_pct":  bias_rel,
            "Mean_SR":   mean_x,
            "N_obs":     len(x),
            "N_pairs":   n_pairs,
        })

df_ccc = pd.DataFrame(ccc_results)


# ── Figures 6A/B/C: per-entity 2x4 scatter — RMSE leads ──────────────────────
entity_letters = dict(zip(ENTITIES.keys(), ["A", "B", "C"]))

for ename, cfg in ENTITIES.items():

    early, late, n_pairs = pair_stores.get(ename, ({c: [] for c, _ in BAND_COLS},
                                                    {c: [] for c, _ in BAND_COLS}, 0))
    if n_pairs == 0:
        print(f"[SKIP] {ename}: no pairs available"); continue

    fig, axes = plt.subplots(2, 4, figsize=(16, 9), sharey=False)
    axes = axes.flatten()

    for idx, (col, label) in enumerate(BAND_COLS):
        ax = axes[idx]
        x = np.array(early[col]);  y = np.array(late[col])
        mask = np.isfinite(x) & np.isfinite(y);  x, y = x[mask], y[mask]

        if len(x) < 10:
            ax.set_title(f"{label} (no data)"); continue

        rmse    = float(np.sqrt(np.mean((y-x)**2)))
        rrmse   = rmse / float(np.mean(x)) * 100
        bias    = float(np.mean(y-x) / np.mean(x) * 100)
        ccc_v   = lin_ccc(x, y)

        ax.scatter(x, y, alpha=0.30, c=cfg["color"], edgecolor="none", s=15)
        vmin = min(x.min(), y.min()) * 0.95
        vmax = max(x.max(), y.max()) * 1.05
        ax.plot([vmin, vmax], [vmin, vmax], "r--", lw=1.5, alpha=0.8)
        ax.set_xlim(vmin, vmax); ax.set_ylim(vmin, vmax); ax.set_aspect("equal")

        # Annotation: RMSE + rRMSE first, bias second, CCC last (with range note)
        ccc_note = " (low var)" if rrmse < 5 else ""
        ax.text(0.05, 0.96,
                f"RMSE:  {rmse:.4f}\n"
                f"rRMSE: {rrmse:.1f}%\n"
                f"Bias:  {bias:+.1f}%\n"
                f"CCC:   {ccc_v:.3f}{ccc_note}",
                transform=ax.transAxes, fontsize=9, va="top", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="wheat",
                          alpha=0.40, edgecolor="gray"))

        ax.set_title(label, fontweight="bold", pad=6)
        if idx >= 4: ax.set_xlabel("Early acquisition (SR)")
        if idx % 4 == 0: ax.set_ylabel("Late acquisition (SR)")
        ax.grid(True, linestyle="--", alpha=0.4)

    letter = entity_letters[ename]
    plt.suptitle(
        f"Figure 6{letter} — Intra-day Radiometric Concordance: {ename}"
        f"  (n = {n_pairs} pairs)",
        fontsize=13, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    fname = f"Fig6{letter}_CCC_{ename.replace('/', '_').replace(' ', '_')}.png"
    out = os.path.join(OUTPUT_DIR, fname)
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[SAVED] {out}")


# ── Figure 6D: cross-entity comparison — relative RMSE | Bias | CCC ──────────
band_order  = [l for _, l in BAND_COLS]
entity_list = list(ENTITIES.keys())
colors      = [ENTITIES[e]["color"] for e in entity_list]

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
n_ent = len(entity_list)
x_pos = np.arange(len(band_order))
w     = 0.70 / n_ent

def get_val(entity, band, metric):
    row = df_ccc[(df_ccc["Entity"] == entity) & (df_ccc["Band"] == band)]
    return row[metric].values[0] if len(row) > 0 else np.nan

# Panel 1: Relative RMSE (normalised — cross-entity comparable)
ax = axes[0]
for k, (ent, col) in enumerate(zip(entity_list, colors)):
    vals = [get_val(ent, b, "rel_RMSE") for b in band_order]
    offset = (k - (n_ent-1)/2) * w
    ax.bar(x_pos + offset, vals, width=w, color=col, alpha=0.78,
           label=ent, edgecolor="white", linewidth=0.5)
ax.set_xticks(x_pos); ax.set_xticklabels(band_order, rotation=40, ha="right")
ax.set_ylabel("Relative RMSE (% of mean SR)", fontweight="bold")
ax.set_title("(D1)  Relative RMSE  [cross-entity comparable]")
ax.legend(fontsize=9)

# Panel 2: Relative Bias
ax = axes[1]
for k, (ent, col) in enumerate(zip(entity_list, colors)):
    vals = [get_val(ent, b, "Bias_pct") for b in band_order]
    offset = (k - (n_ent-1)/2) * w
    ax.bar(x_pos + offset, vals, width=w, color=col, alpha=0.78,
           label=ent, edgecolor="white", linewidth=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x_pos); ax.set_xticklabels(band_order, rotation=40, ha="right")
ax.set_ylabel("Relative Bias (%)", fontweight="bold")
ax.set_title("(D2)  Systematic Radiometric Bias (Late − Early)")
ax.legend(fontsize=9)

# Panel 3: CCC (with range-compression note)
ax = axes[2]
for k, (ent, col) in enumerate(zip(entity_list, colors)):
    vals = [get_val(ent, b, "CCC") for b in band_order]
    offset = (k - (n_ent-1)/2) * w
    ax.bar(x_pos + offset, vals, width=w, color=col, alpha=0.78,
           label=ent, edgecolor="white", linewidth=0.5)
ax.axhline(0.75, color="#555", linestyle="--", linewidth=1,
           alpha=0.6, label="CCC = 0.75")
ax.set_ylim(0, 1.08)
ax.set_xticks(x_pos); ax.set_xticklabels(band_order, rotation=40, ha="right")
ax.set_ylabel("Lin's CCC", fontweight="bold")
ax.set_title("(D3)  Lin's CCC  [depressed by canopy homogeneity in Avocado]")
ax.text(0.02, 0.97,
        "Note: low CCC in Avocado reflects low spatial\n"
        "variance (homogeneous canopy), not poor\n"
        "radiometric consistency. Compare rRMSE (D1).",
        transform=ax.transAxes, fontsize=9, va="top",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#FDFEFE",
                  alpha=0.90, edgecolor="#aaa"))
ax.legend(fontsize=9)

plt.suptitle("Figure 6D — Cross-Entity Spectral Concordance Comparison",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
out6d = os.path.join(OUTPUT_DIR, "Fig6D_CCC_CrossEntity_Comparison.png")
plt.savefig(out6d, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out6d}")

# Summary table
print("\n" + "="*72)
print("SPECTRAL CONCORDANCE SUMMARY (leads: rRMSE, Bias | secondary: CCC)")
print("="*72)
display(
    df_ccc[["Entity","Band","rel_RMSE","Bias_pct","RMSE","CCC","N_pairs"]]
    .rename(columns={"rel_RMSE":"rRMSE (%)","Bias_pct":"Bias (%)","N_pairs":"n pairs"})
    .round(4)
)
df_ccc.to_csv(os.path.join(OUTPUT_DIR, "Table_SpectralConcordance.csv"), index=False)


## Comprobando si las adquisiciones dle mismo día son del mismo satélite o no:

In [ ]:
# ==============================================================================
# SECTION 4B — ORBITAL DRIFT SATELLITE CONSISTENCY CHECK
# ==============================================================================
# Extracts Planet satellite ID from original XML metadata (serialIdentifier)
# and checks whether intra-day drift pairs belong to the same satellite.
# ==============================================================================

import os
import glob
import pandas as pd
import xml.etree.ElementTree as ET


# ------------------------------------------------------------------------------
# 1. Find Metadata_Master_NOAA files
# ------------------------------------------------------------------------------

print("Searching Metadata_Master_NOAA.csv files...")


search_root = os.path.dirname(
    list(PROJECTS.values())[0]["pixel_file"]
)


metadata_files = glob.glob(
    os.path.join(
        search_root,
        "..",
        "**",
        "Metadata_Master_NOAA.csv"
    ),
    recursive=True
)


print(f"Metadata files found: {len(metadata_files)}")

for f in metadata_files:
    print(" ", f)



# ------------------------------------------------------------------------------
# 2. Load all metadata projects
# ------------------------------------------------------------------------------

all_meta = []


for f in metadata_files:

    tmp = pd.read_csv(f)

    tmp["Metadata_Source"] = os.path.basename(
        os.path.dirname(f)
    )

    all_meta.append(tmp)


df_meta = pd.concat(
    all_meta,
    ignore_index=True
)


print("\nMetadata rows:", len(df_meta))


if "XML_Path" not in df_meta.columns:

    raise ValueError(
        "XML_Path missing. Cannot recover satellite metadata."
    )



# ------------------------------------------------------------------------------
# 3. Extract serialIdentifier from XML
# ------------------------------------------------------------------------------

def get_planet_satellite(xml_path):

    if not os.path.exists(xml_path):
        return "XML_NOT_FOUND"

    try:

        root = ET.parse(
            xml_path
        ).getroot()


        for elem in root.iter():

            tag = (
                elem.tag
                .split("}")[-1]
            )


            if tag == "serialIdentifier":

                return (
                    elem.text.strip()
                    if elem.text
                    else None
                )


        return "SERIAL_NOT_FOUND"


    except Exception:

        return "XML_ERROR"



print("\nExtracting satellite IDs from XML...")


df_meta["Satellite_ID"] = (
    df_meta["XML_Path"]
    .apply(get_planet_satellite)
)



print("\nSatellite distribution:")

display(
    df_meta["Satellite_ID"]
    .value_counts()
    .reset_index()
)



# ------------------------------------------------------------------------------
# 4. Merge satellite info into current df_images
# ------------------------------------------------------------------------------

sat_lookup = (
    df_meta[
        [
            "Smart_ID",
            "Satellite_ID"
        ]
    ]
    .drop_duplicates()
)



df_images = (
    df_images
    .drop(
        columns=["Satellite_ID"],
        errors="ignore"
    )
    .merge(
        sat_lookup,
        on="Smart_ID",
        how="left"
    )
)



print(
    "\nImages matched:",
    df_images["Satellite_ID"]
    .notna()
    .sum(),
    "/",
    len(df_images)
)



# ------------------------------------------------------------------------------
# 5. Check intra-day orbital drift pairs
# ------------------------------------------------------------------------------

records = []


for (project, date), group in df_images.groupby(
    [
        "Project",
        "Date_only"
    ]
):

    imgs = (
        group
        .sort_values(
            "True_Solar_Hour"
        )
        [
            [
                "Smart_ID",
                "True_Solar_Hour",
                "Satellite_ID"
            ]
        ]
        .drop_duplicates()
    )


    if len(imgs) < 2:
        continue


    early = imgs.iloc[0]
    late = imgs.iloc[-1]


    delta = (
        late["True_Solar_Hour"]
        -
        early["True_Solar_Hour"]
    )


    if delta < 40/60:
        continue


    records.append(
        {

            "Project": project,

            "Date": date,

            "Early_Image":
                early["Smart_ID"],

            "Late_Image":
                late["Smart_ID"],

            "Delta_TST_hours":
                delta,

            "Early_Satellite":
                early["Satellite_ID"],

            "Late_Satellite":
                late["Satellite_ID"],

            "Same_Satellite":
                early["Satellite_ID"]
                ==
                late["Satellite_ID"]

        }
    )



df_satellite_validation = pd.DataFrame(
    records
)



# ------------------------------------------------------------------------------
# 6. Report results
# ------------------------------------------------------------------------------

print(
"""
====================================================
ORBITAL DRIFT SATELLITE CONSISTENCY CHECK
====================================================
"""
)


display(
    df_satellite_validation
)



if len(df_satellite_validation) > 0:


    same_pct = (
        df_satellite_validation["Same_Satellite"]
        .mean()
        *
        100
    )


    print(
        f"\nSame satellite pairs: {same_pct:.1f}%"
    )


    print("\nSummary:")

    display(
        df_satellite_validation[
            "Same_Satellite"
        ]
        .value_counts()
    )


    if same_pct == 100:

        print(
            """
INTERPRETATION:
All orbital drift intra-day comparisons were acquired
by the same PlanetScope satellite.

Inter-satellite calibration effects are excluded.
"""
        )

    else:

        print(
            """
INTERPRETATION:
Some intra-day comparisons involve different satellites.

PlanetScope L3B harmonization minimizes inter-platform
radiometric differences. Remaining effects are expected
to be secondary compared with illumination geometry.
"""
        )



# ------------------------------------------------------------------------------
# 7. Save evidence table
# ------------------------------------------------------------------------------

out = os.path.join(
    OUTPUT_DIR,
    "Supplementary_SatelliteConsistency_OrbitalDrift.csv"
)


df_satellite_validation.to_csv(
    out,
    index=False
)


print("\nSaved:")
print(out)

---
## Section 4C — Inter-satellite vs orbital-drift: **availability diagnostic**

**Antes** de cuantificar el efecto inter-satélite hay que comprobar cuántos
pares intradía existen en cada régimen. La lógica del contraste es:

| Nivel | Definición | Qué aísla |
|-------|-----------|-----------|
| **N1** mismo-sat / Δt corto (≤10 min) | misma plataforma, hora solar casi idéntica | ruido base de re-adquisición instrumental |
| **N2** distinto-sat / Δt corto (≤10 min) | plataformas distintas, hora solar casi idéntica | **efecto satélite puro** (drift ≈ 0) |
| **N3** distinto-sat / Δt largo (≥40 min) | plataformas distintas, hora solar distinta | satélite **+** orbital drift |

Si **N2** tiene casos suficientes y su diferencia de banda es despreciable
frente a **N3**, todo el peso de la señal en N3 es orbital drift.
Esta celda **solo cuenta**: el propio *n* decide si el análisis formal procede.


In [ ]:
# ==============================================================================
# SECTION 4C — AVAILABILITY DIAGNOSTIC: intra-day pairs by satellite x dt regime
# ==============================================================================
# Requires df_meta / sat_lookup from Section 4B (cell above).
# Produces df_pairs_diag: one row per ordered intra-day image pair, labelled.
# ==============================================================================

DT_SHORT_MIN = 10      # |dt| <= 10 min  -> drift negligible
DT_LONG_MIN  = 40      # |dt| >= 40 min  -> drift regime (matches rest of paper)
MIN_COMMON_PX = 5

# --- 1. Bring Satellite_ID down to PIXEL level -------------------------------
if "Satellite_ID" not in df_pixels.columns:
    df_pixels = df_pixels.merge(sat_lookup, on="Smart_ID", how="left")

n_unmatched = df_pixels["Satellite_ID"].isna().sum()
print(f"Pixels without Satellite_ID after merge: {n_unmatched:,} / {len(df_pixels):,}")
print("Satellite_ID values present in pixel table:")
print(df_pixels["Satellite_ID"].value_counts(dropna=False).to_string())

# --- 2. One row per image (Smart_ID) with its solar time + satellite ----------
img_lvl = (
    df_pixels
    .groupby(["Project", "AOI", "Date_only", "Smart_ID"])
    .agg(TST=("True_Solar_Hour", "first"),
         Satellite=("Satellite_ID", "first"))
    .reset_index()
)

# --- 3. Enumerate ALL intra-day image pairs (not just min/max) ----------------
from itertools import combinations
pair_recs = []
for (proj, aoi, date), g in img_lvl.groupby(["Project", "AOI", "Date_only"]):
    g = g.sort_values("TST")
    if len(g) < 2:
        continue
    for a, b in combinations(g.itertuples(index=False), 2):
        dt_min = abs(b.TST - a.TST) * 60.0
        same_sat = (a.Satellite == b.Satellite)
        if dt_min <= DT_SHORT_MIN:
            level = "N1_sameSat_short" if same_sat else "N2_diffSat_short"
        elif dt_min >= DT_LONG_MIN:
            level = "N3_diffSat_long" if not same_sat else "N3b_sameSat_long"
        else:
            level = "mid (10<dt<40)"
        pair_recs.append({
            "Project": proj, "AOI": aoi, "Date": date,
            "Img_early": a.Smart_ID, "Img_late": b.Smart_ID,
            "Sat_early": a.Satellite, "Sat_late": b.Satellite,
            "dt_min": round(dt_min, 1),
            "Same_Satellite": same_sat,
            "Level": level,
        })

df_pairs_diag = pd.DataFrame(pair_recs)

# --- 4. Report counts per level (THE result we are after) ---------------------
print("\n" + "=" * 64)
print("INTRA-DAY PAIR AVAILABILITY BY REGIME")
print("=" * 64)
if df_pairs_diag.empty:
    print("No intra-day pairs found.")
else:
    summary = (df_pairs_diag
               .groupby(["Project", "Level"])
               .size().reset_index(name="n_pairs")
               .pivot(index="Project", columns="Level", values="n_pairs")
               .fillna(0).astype(int))
    display(summary)
    print("\nGlobal counts per level:")
    print(df_pairs_diag["Level"].value_counts().to_string())

    n_n2 = int((df_pairs_diag["Level"] == "N2_diffSat_short").sum())
    print("\n" + "-" * 64)
    if n_n2 == 0:
        print("[!] N2 (diff-sat, dt<=10min) = 0 pairs.")
        print("    -> Pure inter-satellite contrast NOT available.")
        print("    -> Report this as a data limitation; rely on N3 + harmonization argument.")
    elif n_n2 < 10:
        print(f"[~] N2 = {n_n2} pairs (LOW). Descriptive only — no significance tests.")
        print("    Report median band differences + range; emphasise order-of-magnitude vs N3.")
    else:
        print(f"[OK] N2 = {n_n2} pairs. Enough for descriptive + non-parametric comparison.")
    print("-" * 64)

# expose dt threshold constants for the next cell
INTERSAT_DT_SHORT_MIN = DT_SHORT_MIN
INTERSAT_MIN_COMMON_PX = MIN_COMMON_PX


---
## Section 4D — Inter-satellite band differences (N2) vs orbital drift (N3)

Per-band reflectance differences for matched pixels, computed **only** on the
regimes that the availability diagnostic above found populated. Focus on the
**visible bands and Coastal Blue**, where inter-sensor response differences are
largest. Conteos primero; tests no paramétricos **solo** si n ≥ 10.


In [ ]:
# ==============================================================================
# SECTION 4D — Per-band differences: N2 (diff-sat, short dt) vs N3 (diff-sat, long dt)
# ==============================================================================
# For every qualifying pair, match pixels by pixel_id and compute per-band
# relative difference (late - early)/early * 100. Aggregate per level & band.
# ==============================================================================

INTERSAT_BANDS = ["Coastal_Blue", "Blue", "Green_I", "Green", "Yellow",
                  "Red", "RedEdge", "NIR"]   # visible + coastal first by design

# def per_pair_band_diff(row):
#     """Return dict band -> median relative diff (%) for one pair, or None."""
#     day = df_pixels[(df_pixels["Project"] == row["Project"]) &
#                     (df_pixels["AOI"] == row["AOI"]) &
#                     (df_pixels["Date_only"] == row["Date"])]
#     e = day[day["Smart_ID"] == row["Img_early"]].set_index("pixel_id")
#     l = day[day["Smart_ID"] == row["Img_late"]].set_index("pixel_id")
#     common = e.index.intersection(l.index)
#     if len(common) < INTERSAT_MIN_COMMON_PX:
#         return None
#     out = {"n_px": len(common)}
#     for b in INTERSAT_BANDS:
#         if b in e.columns and b in l.columns:
#             xe = e.loc[common, b].values.astype(float)
#             xl = l.loc[common, b].values.astype(float)
#             mu = np.nanmean(xe)
#             out[b] = float(np.nanmedian(xl - xe) / mu * 100) if mu > 1e-9 else np.nan
#     return out

day_groups = {k: v for k, v in df_pixels.groupby(["Project", "AOI", "Date_only"])}
def per_pair_band_diff(row):
    """Return dict band -> median relative diff (%) for one pair, or None."""
    day = day_groups.get((row["Project"], row["AOI"], row["Date"]))
    if day is None:
        return None
    e = day[day["Smart_ID"] == row["Img_early"]].set_index("pixel_id")
    l = day[day["Smart_ID"] == row["Img_late"]].set_index("pixel_id")
    common = e.index.intersection(l.index)
    if len(common) < INTERSAT_MIN_COMMON_PX:
        return None
    out = {"n_px": len(common)}
    for b in INTERSAT_BANDS:
        if b in e.columns and b in l.columns:
            xe = e.loc[common, b].to_numpy(float)
            xl = l.loc[common, b].to_numpy(float)
            mu = np.nanmean(xe)
            out[b] = float(np.nanmedian(xl - xe) / mu * 100) if mu > 1e-9 else np.nan
    return out

results = []
for level in ["N1_sameSat_short", "N2_diffSat_short", "N3_diffSat_long"]:
    sub = df_pairs_diag[df_pairs_diag["Level"] == level]
    for _, row in sub.iterrows():
        d = per_pair_band_diff(row)
        if d is None:
            continue
        rec = {"Level": level, "Project": row["Project"], "dt_min": row["dt_min"],
               "n_px": d["n_px"]}
        for b in INTERSAT_BANDS:
            rec[b] = d.get(b, np.nan)
        results.append(rec)

df_band_diff = pd.DataFrame(results)

if df_band_diff.empty:
    print("No qualifying pairs with enough common pixels. Nothing to compare.")
else:
    print("Pairs contributing per level:")
    print(df_band_diff["Level"].value_counts().to_string())

    # --- Aggregate: median |relative diff| per band per level -----------------
    agg_rows = []
    for level, g in df_band_diff.groupby("Level"):
        for b in INTERSAT_BANDS:
            vals = g[b].dropna().values
            if len(vals) == 0:
                continue
            agg_rows.append({
                "Level": level, "Band": b, "n_pairs": len(vals),
                "median_diff_pct": round(float(np.median(vals)), 3),
                "median_abs_diff_pct": round(float(np.median(np.abs(vals))), 3),
                "p05": round(float(np.percentile(vals, 5)), 3),
                "p95": round(float(np.percentile(vals, 95)), 3),
            })
    df_intersat_summary = pd.DataFrame(agg_rows)
    print("\n" + "=" * 72)
    print("BAND DIFFERENCE BY LEVEL  (N2 = pure satellite effect; N3 = satellite+drift)")
    print("=" * 72)
    display(df_intersat_summary)

    # --- Headline contrast: |diff| in N2 vs N3 per band -----------------------
    piv = (df_intersat_summary
           .pivot_table(index="Band", columns="Level", values="median_abs_diff_pct"))
    piv = piv.reindex(INTERSAT_BANDS)
    print("\nMedian |relative diff| (%) — satellite effect (N2) vs satellite+drift (N3):")
    display(piv)

    # --- Optional non-parametric test, ONLY if N2 has >= 10 pairs -------------
    from scipy.stats import mannwhitneyu
    n2 = df_band_diff[df_band_diff["Level"] == "N2_diffSat_short"]
    n3 = df_band_diff[df_band_diff["Level"] == "N3_diffSat_long"]
    if len(n2) >= 10 and len(n3) >= 10:
        print("\nMann-Whitney U (|diff| N2 vs N3), per band:")
        for b in INTERSAT_BANDS:
            a = n2[b].dropna().abs().values
            c = n3[b].dropna().abs().values
            if len(a) >= 5 and len(c) >= 5:
                _, p = mannwhitneyu(a, c, alternative="two-sided")
                print(f"  {b:14s} p={p:.4f}  (N2 med={np.median(a):.2f}% | N3 med={np.median(c):.2f}%)")
    else:
        print(f"\n[i] N2={len(n2)} / N3={len(n3)} pairs -> descriptive comparison only "
              f"(no significance tests).")

    df_intersat_summary.to_csv(
        os.path.join(OUTPUT_DIR, "Table_InterSatellite_vs_Drift.csv"), index=False)
    print("\n[SAVED] Table_InterSatellite_vs_Drift.csv")


---
## 5  NDVI Orbital Drift Attribution

**Primary method: bootstrap intra-day regression (\u0394NDVI ~ \u0394TST)**
Same-day pixel pairs control phenology; NDVI difference is driven purely
by illumination change. OLS through origin gives \u03b2 (NDVI h\u207b\u00b9) with bootstrap CI.
Net drift = \u03b2 \u00d7 total TST span of the time series.

**Denominator:**
- *Corema album* — long-term NDVI trend (OLS slope \u00d7 study duration);
  the drift is expressed as % of the observed vegetation recovery.
- *Avocado / Mango* — seasonal NDVI amplitude (P95 \u2212 P05);
  established crops have no directional recovery trend; drift is expressed
  relative to the natural seasonal envelope.

**Mechanistic corroboration:** analytical propagation of measured NIR/Red
band biases through the NDVI equation (Section 4) — reported alongside
the regression to confirm the sign and order of magnitude.


In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import linregress

VEG_FRACTION_MIN = VF_MIN_GLOBAL # 0.70
GAP_MIN = 40
OUT_DIR = r".\03_figures_&_results\00_paper_comparison"

# ── PART A: denominadores contextuales (sin cambios) ───────────────────────────
denom_val, denom_label = {}, {}
for ename, grp in df_pixels.groupby("Entity"):
    s = grp["NDVI"].dropna()
    seasonal_amp = float(s.quantile(0.95) - s.quantile(0.05))
    if ename == "Corema album":
        daily = grp.groupby("Date_only")["NDVI"].median().reset_index().dropna()
        daily["t"] = (pd.to_datetime(daily["Date_only"]) - pd.to_datetime(daily["Date_only"].min())).dt.days.astype(float)
        slope, *_ = linregress(daily["t"], daily["NDVI"]) if len(daily) >= 10 else (np.nan,)*5
        denom_val[ename]   = slope * float(daily["t"].max()) if len(daily) >= 10 else np.nan
        denom_label[ename] = "long-term NDVI trend (OLS)"
    else:
        denom_val[ename], denom_label[ename] = seasonal_amp, "seasonal NDVI amplitude (P95-P05)"

# ── PART B: ΔNDVI ~ ΔTST + ΔVA A NIVEL DE PAR  (bootstrap remuestreando PARES) ──
def boot_beta_pairs(P, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    X, y = P[["dTST", "dVA"]].values, P["dNDVI"].values          # regresión por el origen
    fit = lambda idx: (np.linalg.lstsq(X[idx], y[idx], rcond=None)[0])[0]   # coef de ΔTST
    base = fit(np.arange(len(P)))
    bs = np.array([fit(rng.integers(0, len(P), len(P))) for _ in range(n_boot)]); bs = bs[np.isfinite(bs)]
    return base, float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

reg_rows, band_bias = [], {}
for ename in ENTITIES:
    sub  = df_pixels[df_pixels["Entity"] == ename]
    pure = sub[sub["Veg_Fraction"] >= VEG_FRACTION_MIN] if "Veg_Fraction" in sub.columns else sub

    pair_recs = []
    nir_e, nir_l, red_e, red_l = [], [], [], []
    for (aoi, date), day in pure.groupby(["AOI", "Date_only"]):
        imgs = day.groupby("Smart_ID")[["True_Solar_Hour", "View_Angle"]].first().sort_values("True_Solar_Hour")
        if len(imgs) < 2: continue
        gap = imgs["True_Solar_Hour"].iloc[-1] - imgs["True_Solar_Hour"].iloc[0]
        if gap * 60 < GAP_MIN: continue
        e = day[day["Smart_ID"] == imgs.index[0]].set_index("pixel_id")
        l = day[day["Smart_ID"] == imgs.index[-1]].set_index("pixel_id")
        common = e.index.intersection(l.index)
        if len(common) < 5: continue
        dndvi = float(np.median(l.loc[common, "NDVI"].values - e.loc[common, "NDVI"].values))   # ΔNDVI mediano del par
        if abs(dndvi) > 0.40: continue
        pair_recs.append({"dTST": float(gap),
                          "dVA": float(imgs["View_Angle"].iloc[-1] - imgs["View_Angle"].iloc[0]),
                          "dNDVI": dndvi})
        nir_e += e.loc[common, "NIR"].tolist(); nir_l += l.loc[common, "NIR"].tolist()
        red_e += e.loc[common, "Red"].tolist(); red_l += l.loc[common, "Red"].tolist()

    P = pd.DataFrame(pair_recs)
    if len(P) < 10:
        print(f"[SKIP] {ename}: solo {len(P)} pares válidos"); continue

    beta, lo, hi = boot_beta_pairs(P)
    span = float(sub.groupby("Smart_ID")["True_Solar_Hour"].first().agg(lambda x: x.max() - x.min()))
    net, net_lo, net_hi = beta*span, lo*span, hi*span
    dv, dl = denom_val.get(ename, np.nan), denom_label.get(ename, "")
    rel = abs(net)/abs(dv)*100 if (np.isfinite(dv) and abs(dv) > 1e-6) else np.nan
    sig = "n.s. (IC cruza 0)" if lo*hi <= 0 else "sig"
    reg_rows.append({"Entity": ename, "n_pairs": len(P), "beta_NDVI_per_h": round(beta, 5),
                     "CI95_lo": round(lo, 5), "CI95_hi": round(hi, 5), "TST_span_h": round(span, 2),
                     "Net_NDVI_drift": round(net, 4), "Net_CI_lo": round(net_lo, 4), "Net_CI_hi": round(net_hi, 4),
                     "Denominator": round(dv, 4), "Denom_type": dl,
                     "Drift_pct": round(rel, 1) if np.isfinite(rel) else np.nan, "Significance": sig})
    b = lambda a, c: (np.mean(c) - np.mean(a)) / np.mean(a) * 100
    band_bias[ename] = {"NIR": b(np.array(nir_e), np.array(nir_l)),
                        "Red": b(np.array(red_e), np.array(red_l)),
                        "NDVI0": float(pure["NDVI"].median())}
    print(f"[{ename}] n_pares={len(P)} | beta={beta:+.4f} [{lo:+.4f},{hi:+.4f}]/h | span={span:.2f}h "
          f"| deriva neta={net:+.4f} [{net_lo:+.4f},{net_hi:+.4f}] | {sig}")

reg = pd.DataFrame(reg_rows)
print("\n=== PRIMARIO: regresión intradía A NIVEL DE PAR (IC honesto) ===")
print(reg.to_string(index=False))

# ── PART C: corroboración por propagación analítica de bias de banda ──────────
def ndvi_mech(nir_pct, red_pct, ndvi0):
    red = 1.0; nir = (1 + ndvi0) / (1 - ndvi0)
    nd1 = (nir*(1+nir_pct/100) - red*(1+red_pct/100)) / (nir*(1+nir_pct/100) + red*(1+red_pct/100))
    return nd1 - ndvi0
mech = pd.DataFrame([{"Entity": e, "NIR_bias_pct": round(v["NIR"], 2), "Red_bias_pct": round(v["Red"], 2),
                      "Baseline_NDVI": round(v["NDVI0"], 4),
                      "delta_NDVI_mech": round(ndvi_mech(v["NIR"], v["Red"], v["NDVI0"]), 4)}
                     for e, v in band_bias.items()])
print("\n=== CORROBORACIÓN: propagación analítica (mecanismo NIR/Red) ===")
print(mech.to_string(index=False))
comp = reg[["Entity","Net_NDVI_drift","Net_CI_lo","Net_CI_hi"]].merge(mech[["Entity","delta_NDVI_mech"]], on="Entity")
print("\n=== primario (regresión) vs corroboración (propagación), NDVI absoluto ===")
print(comp.to_string(index=False))

os.makedirs(OUT_DIR, exist_ok=True)
reg.to_csv(os.path.join(OUT_DIR, "Table_NDVI_Drift_Regression.csv"), index=False)
mech.to_csv(os.path.join(OUT_DIR, "Table_NDVI_Drift_Mechanism.csv"), index=False)

# ── Forest plot: deriva neta ± IC (par) + propagación como cuadrado ───────────
fig, ax = plt.subplots(figsize=(8, 4)); order = reg["Entity"].tolist(); yp = np.arange(len(order))
ax.errorbar(reg["Net_NDVI_drift"], yp,
            xerr=[reg["Net_NDVI_drift"]-reg["Net_CI_lo"], reg["Net_CI_hi"]-reg["Net_NDVI_drift"]],
            fmt="o", capsize=4, color="#264653", label="regresión (IC par)")
ax.scatter(mech["delta_NDVI_mech"], [order.index(e) for e in mech["Entity"]],
           marker="s", color="#e76f51", zorder=5, label="propagación")
ax.axvline(0, ls="--", c="grey", alpha=0.7)
ax.set_yticks(yp); ax.set_yticklabels(order); ax.invert_yaxis()
ax.set_xlabel("Deriva NDVI neta sobre la serie (unidades NDVI)")
ax.set_title("Sesgo NDVI por deriva orbital — bootstrap a nivel de par (IC 95%)")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "Fig_NDVI_Drift_ForestPlot.png"), dpi=300, bbox_inches="tight"); plt.show()


---
### Regression by fc regime — pure (fc≥0.70) vs soil (fc≤0.30)

Control by contrast for the cancellation claim: the per-pair drift slope
β(ΔNDVI/h) and its bootstrap CI are recomputed **per fc regime**. Expectation:
β and its CI grow as fc drops (pure pixels cancel, soil pixels do not).
n_pairs per regime is reported — **do not over-read regimes with n<10.**


In [ ]:
# ============================================================
# [NOT USABLE FOR PAPER] fc-regime pure-vs-soil contrast.
# Reason: AOIs were delineated over vegetation, so soil/mixed
# pixels are too few (27/53/223 unique px at fc<=0.50) and the
# pair-level bootstrap resamples DAYS, not pixels -> pure & soil
# end up with near-identical n_pairs and beta. Use the Purity
# Sweep (continuous, pixel-level) as the purity evidence instead.
# Kept only for traceability. Do NOT cite these numbers.
# ============================================================

# ==============================================================================
# fc-REGIME CONTRAST — per-pair ΔNDVI ~ ΔTST + ΔVA, bootstrap CI, by regime
# Reuses boot_beta_pairs() defined in the PART B cell above.
# ==============================================================================
FC_REGIMES = {
    "pure (fc>=0.70)": (0.70, 1.01),
    "soil (fc<=0.30)": (0.00, 0.30),
}
GAP_MIN_RG = 40

def pairs_for_regime(sub, lo, hi):
    band = sub[(sub["Veg_Fraction"] >= lo) & (sub["Veg_Fraction"] < hi)] \
        if "Veg_Fraction" in sub.columns else sub
    recs = []
    for (aoi, date), day in band.groupby(["AOI", "Date_only"]):
        imgs = day.groupby("Smart_ID")[["True_Solar_Hour", "View_Angle"]] \
                  .first().sort_values("True_Solar_Hour")
        if len(imgs) < 2:
            continue
        gap = imgs["True_Solar_Hour"].iloc[-1] - imgs["True_Solar_Hour"].iloc[0]
        if gap * 60 < GAP_MIN_RG:
            continue
        e = day[day["Smart_ID"] == imgs.index[0]].set_index("pixel_id")
        l = day[day["Smart_ID"] == imgs.index[-1]].set_index("pixel_id")
        common = e.index.intersection(l.index)
        if len(common) < 5:
            continue
        dndvi = float(np.median(l.loc[common, "NDVI"].values -
                                e.loc[common, "NDVI"].values))
        if abs(dndvi) > 0.40:
            continue
        recs.append({"dTST": float(gap),
                     "dVA": float(imgs["View_Angle"].iloc[-1] - imgs["View_Angle"].iloc[0]),
                     "dNDVI": dndvi})
    return pd.DataFrame(recs)

regime_rows = []
print("=" * 78)
print("fc-REGIME DRIFT CONTRAST")
print("=" * 78)
for ename in ENTITIES:
    sub = df_pixels[df_pixels["Entity"] == ename]
    for rname, (lo, hi) in FC_REGIMES.items():
        P = pairs_for_regime(sub, lo, hi)
        if len(P) < 10:
            print(f"  [skip] {ename:14s} {rname:18s}  n_pairs={len(P)}  (insufficient)")
            regime_rows.append({"Entity": ename, "Regime": rname,
                                "n_pairs": len(P), "beta_per_h": np.nan,
                                "CI_lo": np.nan, "CI_hi": np.nan})
            continue
        beta, lo_ci, hi_ci = boot_beta_pairs(P)
        sig = "n.s." if lo_ci * hi_ci <= 0 else "sig"
        print(f"  {ename:14s} {rname:18s}  n_pairs={len(P):3d}  "
              f"beta={beta:+.4f}/h  CI95=[{lo_ci:+.4f},{hi_ci:+.4f}]  {sig}")
        regime_rows.append({"Entity": ename, "Regime": rname, "n_pairs": len(P),
                            "beta_per_h": round(beta, 5),
                            "CI_lo": round(lo_ci, 5), "CI_hi": round(hi_ci, 5),
                            "Significance": sig})

df_regime = pd.DataFrame(regime_rows)
df_regime.to_csv(os.path.join(OUTPUT_DIR, "Table_fcRegime_DriftContrast.csv"), index=False)
print("\n[SAVED] Table_fcRegime_DriftContrast.csv")
display(df_regime)


---
## 5B.  Purity Sweep & PRI Vulnerability

This section tests two claims empirically using the same intra-day pairs,
sweeping the `Veg_Fraction` threshold from 0 % (all pixels) to 90 % (pure pixels):

1. **Apparent drift is soil-driven.**  Red-band bias in Avocado falls from ~+7 % at 0 %
   purity to ~-3 % at 70 %, confirming that soil contamination (bright in Red, variable
   with illumination) was generating the spurious signal.

2. **NDVI is robust by mechanism.**  NIR and Red drift together; at high purity their
   biases become similar in sign and magnitude and cancel in the ratio.
   NDVI CCC improves monotonically with purity.

3. **PRI is structurally vulnerable.**  Green I (531 nm) and Yellow (570 nm) drift
   *asymmetrically* at every purity level: they amplify rather than cancel in the ratio.
   PRI concordance stays low even at 70 % purity.

> **Outputs saved to** `00_paper_comparison/`:
> `Table_PuritySweep.csv`, `Fig_PuritySweep_BandBias.png`,
> `Fig_PuritySweep_NDVI_vs_PRI.png`


In [ ]:
# ==============================================================================
# 5B. PURITY SWEEP + PRI VULNERABILITY
# ==============================================================================

THRESHOLDS_VF = [0.0, 0.20, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
SWEEP_BANDS   = ["Coastal_Blue","Blue","Green_I","Green","Yellow",
                 "Red","RedEdge","NIR","NDVI","PRI_proxy"]
GAP_H_SWEEP   = 40 / 60.0   # hours


# def concordance_sweep(df_ent, vf_min, bands, gap_h=GAP_H_SWEEP):
#     filt = (df_ent[df_ent["Veg_Fraction"] >= vf_min]
#             if ("Veg_Fraction" in df_ent.columns and vf_min > 0) else df_ent)
#     early = {b: [] for b in bands}
#     late  = {b: [] for b in bands}
#     n_pairs = 0
#     for (aoi, date), day in filt.groupby(["AOI", "Date_only"]):
#         imgs = day.groupby("Smart_ID")["True_Solar_Hour"].first().sort_values()
#         if len(imgs) < 2: continue
#         if imgs.iloc[-1] - imgs.iloc[0] < gap_h: continue
#         e = day[day["Smart_ID"] == imgs.index[0]].set_index("pixel_id")
#         l = day[day["Smart_ID"] == imgs.index[-1]].set_index("pixel_id")
#         com = e.index.intersection(l.index)
#         if len(com) < 5: continue
#         for b in bands:
#             if b in e.columns:
#                 early[b].extend(e.loc[com, b].values.tolist())
#                 late[b].extend(l.loc[com, b].values.tolist())
#         n_pairs += 1
#     out = {"n_pairs": n_pairs, "n_pixels": int(filt["pixel_id"].nunique())}
#     for b in bands:
#         x = np.array(early[b], float); y = np.array(late[b], float)
#         ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
#         if len(x) < 10:
#             out[f"bias_{b}"] = np.nan
#             out[f"rrmse_{b}"] = np.nan
#             out[f"ccc_{b}"] = np.nan
#         else:
#             mu  = max(float(np.mean(x)), 1e-10)
#             out[f"bias_{b}"]  = round(float(np.mean(y-x)) / mu * 100, 3)
#             out[f"rrmse_{b}"] = round(float(np.sqrt(np.mean((y-x)**2))) / mu * 100, 3)
#             r   = float(np.corrcoef(x, y)[0, 1])
#             st  = float(np.std(x)); sp = float(np.std(y))
#             den = x.var() + y.var() + (float(np.mean(x)) - float(np.mean(y)))**2
#             out[f"ccc_{b}"] = round((2*r*st*sp)/den if den > 0 else np.nan, 4)
#     return out

def concordance_sweep(df_ent, vf_min, bands, gap_h=GAP_H_SWEEP):
    filt = (df_ent[df_ent["Veg_Fraction"] >= vf_min]
            if ("Veg_Fraction" in df_ent.columns and vf_min > 0) else df_ent)
    early = {b: [] for b in bands}
    late  = {b: [] for b in bands}
    n_pairs = 0
    
    for (aoi, date), day in filt.groupby(["AOI", "Date_only"]):
        imgs = day.groupby("Smart_ID")["True_Solar_Hour"].first().sort_values()
        if len(imgs) < 2: continue
        if imgs.iloc[-1] - imgs.iloc[0] < gap_h: continue
        
        e = day[day["Smart_ID"] == imgs.index[0]].set_index("pixel_id")
        l = day[day["Smart_ID"] == imgs.index[-1]].set_index("pixel_id")
        com = e.index.intersection(l.index)
        if len(com) < 5: continue
        
        # Filtro QC (Celda 12): Descartar pares con salto no físico por bruma/nubes
        if "NDVI" in e.columns and "NDVI" in l.columns:
            delta_ndvi = l.loc[com, "NDVI"] - e.loc[com, "NDVI"]
            if abs(delta_ndvi.median()) > 0.40:
                continue
                
        for b in bands:
            if b in e.columns:
                early[b].extend(e.loc[com, b].values.tolist())
                late[b].extend(l.loc[com, b].values.tolist())
        n_pairs += 1
        
    out = {"n_pairs": n_pairs, "n_pixels": int(filt["pixel_id"].nunique())}
    
    for b in bands:
        x = np.array(early[b], float); y = np.array(late[b], float)
        ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
        if len(x) < 10:
            out[f"bias_{b}"] = np.nan
            out[f"rrmse_{b}"] = np.nan
            out[f"ccc_{b}"] = np.nan
        else:
            mu  = max(float(np.mean(x)), 1e-10)
            out[f"bias_{b}"]  = round(float(np.mean(y-x)) / mu * 100, 3)
            out[f"rrmse_{b}"] = round(float(np.sqrt(np.mean((y-x)**2))) / mu * 100, 3)
            r   = float(np.corrcoef(x, y)[0, 1])
            st  = float(np.std(x)); sp = float(np.std(y))
            den = x.var() + y.var() + (float(np.mean(x)) - float(np.mean(y)))**2
            out[f"ccc_{b}"] = round((2*r*st*sp)/den if den > 0 else np.nan, 4)
            
    return out


# ── Run sweep ──────────────────────────────────────────────────────────────────
print("Purity sweep running...")
sweep_rows = []

for ename in ENTITIES:
    sub = df_pixels[df_pixels["Entity"] == ename]
    for thresh in THRESHOLDS_VF:
        res = concordance_sweep(sub, thresh, SWEEP_BANDS)
        sweep_rows.append({"Entity": ename,
                           "Threshold_pct": int(round(thresh * 100)), **res})
        print(f"  {ename:16s}  thr={thresh*100:2.0f}%  "
              f"n_pairs={res['n_pairs']:3d}  "
              f"Red={res.get('bias_Red',float('nan')):+.1f}%  "
              f"NIR={res.get('bias_NIR',float('nan')):+.1f}%  "
              f"GI={res.get('bias_Green_I',float('nan')):+.1f}%  "
              f"Green={res.get('bias_Green',float('nan')):+.1f}%  "
              f"NDVI_CCC={res.get('ccc_NDVI',float('nan')):.3f}  "
              f"PRI_CCC={res.get('ccc_PRI_proxy',float('nan')):.3f}")

df_sweep = pd.DataFrame(sweep_rows)
df_sweep.to_csv(os.path.join(OUTPUT_DIR, "Table_PuritySweep.csv"), index=False)
print(f"\n[SAVED] Table_PuritySweep.csv")


# ── Comparison table: 0% vs 70% ───────────────────────────────────────────────
print("\n" + "="*80)
print("KEY COMPARISON: ALL PIXELS (0%) vs PURE PIXELS (70%)")
print("="*80)
cols_cmp = [c for c in ["bias_Red","bias_NIR","bias_Green_I","bias_Green",
                         "ccc_NDVI","ccc_PRI_proxy","rrmse_NDVI","rrmse_PRI_proxy"]
            if c in df_sweep.columns]
comp_tbl = (df_sweep[df_sweep["Threshold_pct"].isin([0, 70])]
            [["Entity","Threshold_pct","n_pixels"] + cols_cmp]
            .sort_values(["Entity","Threshold_pct"]))
display(comp_tbl.round(3))


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE A: Band bias vs purity threshold
# ─────────────────────────────────────────────────────────────────────────────
band_panels = [
    ("bias_Red",     "Red",              "Soil-driver: drops with purity (Avocado)"),
    ("bias_NIR",     "NIR",              "Co-drifts with Red -> NDVI cancellation"),
    ("bias_Green_I", "Green I (531 nm)", "PRI numerator — asymmetric drift"),
    ("bias_Green",  "Green (565 nm)",  "PRI denominator — asymmetric drift"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
axes = axes.flatten()

for ax, (col, band_label, subtitle) in zip(axes, band_panels):
    for ename, cfg in ENTITIES.items():
        sub = df_sweep[df_sweep["Entity"] == ename].sort_values("Threshold_pct")
        if col not in sub.columns: continue
        ax.plot(sub["Threshold_pct"], sub[col],
                color=cfg["color"], marker=cfg["marker"],
                linewidth=2, markersize=7, label=ename)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.7)
    ax.axvline(70, color="black", linestyle=":", linewidth=1.5, alpha=0.45,
               label="70% threshold")
    ax.set_title(f"{band_label}  [{subtitle}]", fontweight="bold", fontsize=10)
    ax.set_ylabel("Relative bias (%)", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=":", alpha=0.5)

for ax in axes[2:]:
    ax.set_xlabel("Vegetation purity threshold (%)")

# plt.suptitle("Figure A — Band Bias vs. Pixel Purity Filter\n"
#              "Soil contamination drives apparent drift in Avocado Red band",
#              fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
out_a = os.path.join(OUTPUT_DIR, "Fig_PuritySweep_BandBias.png")
plt.savefig(out_a, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out_a}")


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE B: NDVI vs PRI concordance + band-pair asymmetry
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(17, 5))

# Panel 1: CCC sweep
ax = axes[0]
for ename, cfg in ENTITIES.items():
    sub = df_sweep[df_sweep["Entity"] == ename].sort_values("Threshold_pct")
    ax.plot(sub["Threshold_pct"], sub.get("ccc_NDVI", pd.Series(dtype=float)),
            color=cfg["color"], marker=cfg["marker"],
            linewidth=2, markersize=6, label=f"{ename} NDVI")
    if "ccc_PRI_proxy" in sub.columns:
        ax.plot(sub["Threshold_pct"], sub["ccc_PRI_proxy"],
                color=cfg["color"], marker=cfg["marker"],
                linewidth=2, markersize=6, linestyle="--", alpha=0.55,
                label=f"{ename} PRI")
ax.axhline(0.75, color="#888", linestyle=":", linewidth=1.2, alpha=0.7)
ax.axvline(70, color="black", linestyle=":", linewidth=1.5, alpha=0.45)
ax.set_xlim(-5, 95); ax.set_ylim(0, 1.08)
ax.set_xlabel("Purity threshold (%)"); ax.set_ylabel("Lin's CCC")
ax.set_title("(A) CCC: NDVI (solid) vs PRI (dashed)", fontweight="bold")
ax.legend(fontsize=8)

# # Panel 2: rRMSE sweep
# ax = axes[1]
# for ename, cfg in ENTITIES.items():
#     sub = df_sweep[df_sweep["Entity"] == ename].sort_values("Threshold_pct")
#     if "rrmse_NDVI" in sub.columns:
#         ax.plot(sub["Threshold_pct"], sub["rrmse_NDVI"],
#                 color=cfg["color"], marker=cfg["marker"],
#                 linewidth=2, markersize=6, label=f"{ename} NDVI")
#     if "rrmse_PRI_proxy" in sub.columns:
#         ax.plot(sub["Threshold_pct"], sub["rrmse_PRI_proxy"],
#                 color=cfg["color"], marker=cfg["marker"],
#                 linewidth=2, markersize=6, linestyle="--", alpha=0.55,
#                 label=f"{ename} PRI")
# ax.axvline(70, color="black", linestyle=":", linewidth=1.5, alpha=0.45)
# ax.set_xlabel("Purity threshold (%)"); ax.set_ylabel("Relative RMSE (%)")
# ax.set_title("(B) rRMSE: NDVI (solid) vs PRI (dashed)", fontweight="bold")
# ax.legend(fontsize=8)

# Panel 3: band-pair asymmetry at 0% vs 70%
ax = axes[1]
for thresh, mk, alpha_v in [(0, "X", 0.55), (70, "D", 1.0)]:
    sub_t = df_sweep[df_sweep["Threshold_pct"] == thresh]
    for ename, cfg in ENTITIES.items():
        row = sub_t[sub_t["Entity"] == ename]
        if row.empty: continue
        r = row.iloc[0]
        nir_red = abs(r.get("bias_NIR", float("nan")) - r.get("bias_Red", float("nan")))
        gi_yel  = abs(r.get("bias_Green_I", float("nan")) - r.get("bias_Green", float("nan")))
        ax.scatter(nir_red, gi_yel, color=cfg["color"], marker=mk,
                   s=130, alpha=alpha_v, edgecolors="white", linewidths=0.8, zorder=4)
        ax.annotate(f"{ename[:6]} {thresh}%", (nir_red, gi_yel),
                    fontsize=7.5, xytext=(3, 3), textcoords="offset points",
                    color=cfg["color"])

ax.set_xlabel("|NIR bias - Red bias|  (%)")
ax.set_ylabel("|Green I bias - Green bias|  (%)")
ax.set_title("(B) Band-pair asymmetry\nX=0%  D=70% purity", fontweight="bold")
ax.text(0.05, 0.97,
        "Left/bottom = good cancellation\n"
        "High Y = PRI contaminated\n(GI/Green asymmetric)",
        transform=ax.transAxes, fontsize=8.5, va="top",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#fafafa",
                  alpha=0.9, edgecolor="#ccc"))
handles = [plt.Line2D([0],[0], color=ENTITIES[e]["color"],
                       marker=ENTITIES[e]["marker"], linewidth=0,
                       markersize=8, label=e) for e in ENTITIES]
ax.legend(handles=handles, fontsize=8)

# plt.suptitle("Figure B — NDVI Robustness vs PRI Vulnerability Across Purity Thresholds",
#              fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
out_b = os.path.join(OUTPUT_DIR, "Fig_PuritySweep_NDVI_vs_PRI.png")
plt.savefig(out_b, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out_b}")

print("\nKey finding:")
print("  NDVI: NIR and Red drift together -> cancel in ratio -> robust at 70% purity")
print("  PRI:  Green_I and Green drift asymmetrically -> remain contaminated at 70%")


---
### Fig 8 — NDVI cancellation mechanism (reads df_sweep @70%)
Moved to run AFTER the Purity Sweep so df_sweep always exists when this runs.

In [ ]:
# Colores de cobertura — FUERA de la paleta band-basis (verde/naranja/rojo)
# para que el color de línea (cobertura) nunca colisione con el color de tick (band-basis).
COVER_PLOT_COLOR = {
    "Corema album": "#0072B2",   # azul
    "Avocado":      "#762A83",   # morado
    "Mango":        "#000000",   # negro
}
# Etiqueta solo-figura (la columna/clave interna sigue siendo PRI_proxy)
DISPLAY_NAME = {"PRI_proxy": "PRIsat"}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==============================================================================
# Fig 8 — NDVI cancellation mechanism
# CCC values are now READ FROM df_sweep at the 70% purity threshold,
# instead of being hardcoded. Requires the Purity Sweep cell (5B) to have run.
# ==============================================================================

entities = ['Corema album', 'Avocado', 'Mango']

def sweep_val(entity, col, thr=70):
    row = df_sweep[(df_sweep["Entity"] == entity) &
                   (df_sweep["Threshold_pct"] == thr)]
    return float(row[col].values[0]) if (len(row) and col in df_sweep.columns) else np.nan

# CCC at 70% read live from the sweep table
ndvi_ccc = np.array([sweep_val(e, "ccc_NDVI")      for e in entities])
pri_ccc  = np.array([sweep_val(e, "ccc_PRI_proxy") for e in entities])

# Band biases at 70% read live as well (Red/NIR) for panel A
red_bias = np.array([sweep_val(e, "bias_Red") for e in entities])
nir_bias = np.array([sweep_val(e, "bias_NIR") for e in entities])

# Net mechanistic delta-NDVI from analytic propagation of those biases
def ndvi_mech(nir_pct, red_pct, ndvi0):
    red = 1.0; nir = (1 + ndvi0) / (1 - ndvi0)
    nd1 = (nir*(1+nir_pct/100) - red*(1+red_pct/100)) / \
          (nir*(1+nir_pct/100) + red*(1+red_pct/100))
    return nd1 - ndvi0

ndvi0_by_ent = {e: float(df_pixels[df_pixels["Entity"] == e]["NDVI"].median())
                for e in entities}
ndvi_mech_arr = np.array([ndvi_mech(nir_bias[i], red_bias[i], ndvi0_by_ent[e])
                          for i, e in enumerate(entities)])

print("Values read live from df_sweep @70% (no hardcoding):")
for i, e in enumerate(entities):
    print(f"  {e:14s} NDVI_CCC={ndvi_ccc[i]:.3f}  PRI_CCC={pri_ccc[i]:.3f}  "
          f"Red={red_bias[i]:+.2f}%  NIR={nir_bias[i]:+.2f}%  "
          f"netDNDVI={ndvi_mech_arr[i]:+.4f}")

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(entities)); w = 0.28

# ----- PANEL A: band biases + net NDVI as annotation -----
axA.bar(x - w/2, red_bias, w, label='Red bias',  color='#d62728')
axA.bar(x + w/2, nir_bias, w, label='NIR bias',  color='#8c564b')
axA.axhline(0, color='grey', lw=0.8)
ymax = np.nanmax(np.abs(np.concatenate([red_bias, nir_bias]))) * 1.25 + 0.5
for i, v in enumerate(ndvi_mech_arr):
    axA.annotate(f'Net ΔNDVI\n{v:+.4f}', (x[i], ymax*0.72), ha='center', va='bottom',
                 fontsize=9, color='#2ca02c', fontweight='bold')
axA.set_xticks(x); axA.set_xticklabels(order, rotation=35, ha="right") #axA.set_xticklabels(entities)
axA.set_ylabel('Band bias (%)')
axA.set_ylim(-ymax, ymax)
axA.set_title('(A) Co-directional band drift cancels in NDVI')
axA.legend(loc='lower left')

# ----- PANEL B: NDVI CCC vs PRI CCC -----
axB.bar(x - w/2, ndvi_ccc, w, label='NDVI CCC', color='#2ca02c')
axB.bar(x + w/2, pri_ccc,  w, label='PRI CCC',  color='#ff7f0e')
axB.axhline(0.75, color='grey', ls='--', lw=1, label='CCC = 0.75')
axB.set_xticks(x); axB.set_xticklabels(entities)
axB.set_ylabel("Lin's CCC (intra-day pairs, fc >= 70%)")
axB.set_ylim(0, 1.0)
# axB.set_title('(B) Drift resilience: NDVI vs PRI')
axB.set_title('(B) Drift resilience: NDVI vs PRI\n'
              'Corema CCC deflated by low spatial variance, not drift (cf. panel A)',
              fontsize=9.5)
axB.legend()

plt.tight_layout()
plt.savefig('Fig8_NDVI_cancellation_mechanism.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")


---
## Section 5C — Multi-index drift robustness (structural ↔ pigment axis)

Indices are computed per pixel from the 8 SuperDove bands, then passed through
the **same intra-day pairing machinery** as NDVI/PRI (fc≥70%, Δt≥40 min,
|ΔNDVI|≤0.40 QC). The hypothesis: drift robustness is ordered by (a) how
co-directionally the constituent bands drift and (b) whether the underlying
biophysical process is slow (structure) or fast (pigment cycling).

**EVI2** is used instead of 3-band EVI to avoid the Blue band (high inter-sat
diff). **CCI** uses Green_I (531 nm, the xanthophyll band) — declare in Methods.


In [ ]:
# ==============================================================================
# SECTION 5C — Multi-index computation + intra-day concordance (CCC/bias/rRMSE)
# ==============================================================================
# Adds derived indices to a working copy of df_pixels, then reuses the
# concordance_sweep() pairing logic to score each index at the 70% threshold.
# ==============================================================================

# --- 1. Compute indices per pixel (vectorised, safe denominators) -------------
def safe_div(num, den):
    den = den.replace(0, np.nan)
    return num / den

dfx = df_pixels.copy()
B = {c: dfx[c].astype(float) for c in
     ["Coastal_Blue","Blue","Green_I","Green","Yellow","Red","RedEdge","NIR"]
     if c in dfx.columns}

# Structural
dfx["EVI2"]  = 2.5 * safe_div(B["NIR"] - B["Red"], B["NIR"] + 2.4*B["Red"] + 1.0)
dfx["NDRE"]  = safe_div(B["NIR"] - B["RedEdge"], B["NIR"] + B["RedEdge"])
dfx["OSAVI"] = safe_div(B["NIR"] - B["Red"], B["NIR"] + B["Red"] + 0.16)
# Mixed
dfx["GNDVI"] = safe_div(B["NIR"] - B["Green"], B["NIR"] + B["Green"])
# Pigment / chlorophyll
dfx["CIre"]  = safe_div(B["NIR"], B["RedEdge"]) - 1.0
dfx["CCI"]   = safe_div(B["Green_I"] - B["Red"], B["Green_I"] + B["Red"])
# NDVI and PRI_proxy already exist in df_pixels

INDEX_ORDER = [
    ("NDVI",      "structural"),
    ("EVI2",      "structural"),
    ("NDRE",      "structural"),
    ("OSAVI",     "structural"),
    ("GNDVI",     "mixed"),
    ("CIre",      "pigment/chl"),
    ("CCI",       "pigment/xanth"),
    ("PRI_proxy", "pigment/xanth"),
]
INDEX_NAMES = [n for n, _ in INDEX_ORDER if n in dfx.columns]
print("Indices available:", INDEX_NAMES)

# --- 2. Intra-day concordance per index (mirrors concordance_sweep at 70%) -----
VF_MIN_IDX = VF_MIN_GLOBAL # 0.70
GAP_H_IDX  = 40 / 60.0

def index_concordance(df_ent, idx_cols, vf_min=VF_MIN_IDX, gap_h=GAP_H_IDX):
    filt = (df_ent[df_ent["Veg_Fraction"] >= vf_min]
            if ("Veg_Fraction" in df_ent.columns and vf_min > 0) else df_ent)
    early = {b: [] for b in idx_cols}
    late  = {b: [] for b in idx_cols}
    n_pairs = 0
    for (aoi, date), day in filt.groupby(["AOI", "Date_only"]):
        imgs = day.groupby("Smart_ID")["True_Solar_Hour"].first().sort_values()
        if len(imgs) < 2:
            continue
        if imgs.iloc[-1] - imgs.iloc[0] < gap_h:
            continue
        e = day[day["Smart_ID"] == imgs.index[0]].set_index("pixel_id")
        l = day[day["Smart_ID"] == imgs.index[-1]].set_index("pixel_id")
        com = e.index.intersection(l.index)
        if len(com) < 5:
            continue
        if "NDVI" in e.columns and "NDVI" in l.columns:
            dn = l.loc[com, "NDVI"] - e.loc[com, "NDVI"]
            if abs(dn.median()) > 0.40:
                continue
        for b in idx_cols:
            if b in e.columns:
                early[b].extend(e.loc[com, b].values.tolist())
                late[b].extend(l.loc[com, b].values.tolist())
        n_pairs += 1
    rows = []
    for b in idx_cols:
        x = np.array(early[b], float); y = np.array(late[b], float)
        ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
        if len(x) < 10:
            rows.append({"Index": b, "n_obs": len(x), "CCC": np.nan,
                         "bias_pct": np.nan, "rrmse_pct": np.nan, "n_pairs": n_pairs})
            continue
        mu = float(np.mean(x))
        # CCC
        r = float(np.corrcoef(x, y)[0, 1])
        st, sp = float(np.std(x)), float(np.std(y))
        den = x.var() + y.var() + (mu - float(np.mean(y)))**2
        ccc = (2*r*st*sp)/den if den > 0 else np.nan
        # bias / rRMSE normalised by |mean| (indices can be near 0)
        scale = max(abs(mu), 1e-6)
        rows.append({
            "Index": b, "n_obs": len(x),
            "CCC": round(ccc, 4),
            "bias_pct": round(float(np.mean(y - x)) / scale * 100, 3),
            "rrmse_pct": round(float(np.sqrt(np.mean((y - x)**2))) / scale * 100, 3),
            "n_pairs": n_pairs,
        })
    return pd.DataFrame(rows)

multi_rows = []
for ename in ENTITIES:
    sub = dfx[dfx["Entity"] == ename]
    res = index_concordance(sub, INDEX_NAMES)
    res["Entity"] = ename
    multi_rows.append(res)

df_multi = pd.concat(multi_rows, ignore_index=True)
type_map = dict(INDEX_ORDER)
df_multi["Type"] = df_multi["Index"].map(type_map)
df_multi = df_multi[["Entity","Index","Type","CCC","bias_pct","rrmse_pct","n_obs","n_pairs"]]

print("\n" + "=" * 78)
print("MULTI-INDEX INTRA-DAY ROBUSTNESS @ fc>=70%  (CCC high = robust to drift)")
print("=" * 78)
display(df_multi.sort_values(["Entity", "CCC"], ascending=[True, False]))

# Pivot: CCC by index x entity, ordered structural -> pigment
piv_ccc = (df_multi.pivot_table(index="Index", columns="Entity", values="CCC")
           .reindex([n for n, _ in INDEX_ORDER]))
print("\nCCC by index (rows ordered structural -> pigment):")
display(piv_ccc)

df_multi.to_csv(os.path.join(OUTPUT_DIR, "Table_MultiIndex_DriftRobustness.csv"), index=False)
print("\n[SAVED] Table_MultiIndex_DriftRobustness.csv")

# --- 3. Band-composition tag (the NEW mechanistic axis) -----------------------
# Robustness is predicted by whether numerator/denominator bands co-drift,
# not by structural-vs-pigment. NIR-anchored ratios cancel; visible-asymmetric
# ratios do not. CIre (chlorophyll, "pigment") is robust BECAUSE it is NIR/RedEdge.
BAND_BASIS = {
    "NDVI":      "NIR-anchored",
    "EVI2":      "NIR-anchored",
    "NDRE":      "NIR-anchored",
    "OSAVI":     "NIR-anchored",
    "CIre":      "NIR-anchored",
    "GNDVI":     "NIR+visible",
    "CCI":       "visible-asymmetric",
    "PRI_proxy": "visible-asymmetric",
}
BASIS_COLOR = {
    "NIR-anchored":       "#2ca02c",
    "NIR+visible":        "#ff7f0e",
    "visible-asymmetric": "#d62728",
}
df_multi["Basis"] = df_multi["Index"].map(BAND_BASIS)

# --- 4. Order indices by MEDIAN rRMSE across entities (variance-independent) ---
# rRMSE normalises by index magnitude, so it is comparable across covers and
# does NOT collapse on low-spatial-variance covers the way CCC does (Corema).
rrmse_med = (df_multi.groupby("Index")["rrmse_pct"].median()
             .sort_values())                      # ascending = most robust first
order = [ix for ix in rrmse_med.index if ix in df_multi["Index"].values]
xp = np.arange(len(order))
bar_colors = [BASIS_COLOR[BAND_BASIS[ix]] for ix in order]

# --- 5. Two-panel figure: (A) rRMSE [log], (B) CCC -----------------------------
fig, (axA, axB) = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)

# Panel A — rRMSE (primary, variance-independent), log scale
for ename, cfg in ENTITIES.items():
    vals = [df_multi[(df_multi["Entity"] == ename) &
                     (df_multi["Index"] == ix)]["rrmse_pct"].values for ix in order]
    vals = [v[0] if len(v) else np.nan for v in vals]
    axA.plot(xp, vals, marker=cfg.get("marker", "o"), color=COVER_PLOT_COLOR.get(ename, cfg.get("color")), #color=cfg.get("color", None),
             linewidth=2, markersize=8, label=ename)
axA.set_yscale("log")
axA.axhline(100, color="grey", ls=":", lw=1, alpha=0.7)
axA.text(len(order) - 1, 108, "rRMSE = 100%\n(error > index magnitude)",
         fontsize=7.5, ha="right", va="bottom", color="grey")
axA.set_xticks(xp)
# axA.set_xticklabels(order, rotation=35, ha="right")
axA.set_xticklabels([DISPLAY_NAME.get(ix, ix) for ix in order], rotation=35, ha="right")

for tick, ix in zip(axA.get_xticklabels(), order):
    tick.set_color(BASIS_COLOR[BAND_BASIS[ix]])
axA.set_ylabel("Intra-day rRMSE (%)  [log]", fontweight="bold")
axA.set_title("(A) Drift error by index — ordered by median rRMSE\n"
              "(lower = more robust; colour = band basis)", fontweight="bold")
axA.legend(fontsize=9, title="Cover")

# Panel B — CCC (secondary; shows where it misleads on low-variance covers)
for ename, cfg in ENTITIES.items():
    vals = [df_multi[(df_multi["Entity"] == ename) &
                     (df_multi["Index"] == ix)]["CCC"].values for ix in order]
    vals = [v[0] if len(v) else np.nan for v in vals]
    axB.plot(xp, vals, marker=cfg.get("marker", "o"), color=COVER_PLOT_COLOR.get(ename, cfg.get("color")), #color=cfg.get("color", None),
             linewidth=2, markersize=8, label=ename)
axB.axhline(0.75, color="grey", ls="--", lw=1, alpha=0.7, label="CCC = 0.75")
axB.set_xticks(xp)
# axB.set_xticklabels(order, rotation=35, ha="right")
axB.set_xticklabels([DISPLAY_NAME.get(ix, ix) for ix in order], rotation=35, ha="right")
for tick, ix in zip(axB.get_xticklabels(), order):
    tick.set_color(BASIS_COLOR[BAND_BASIS[ix]])
axB.set_ylim(0, 1.05)
axB.set_ylabel("Lin's CCC (intra-day, fc>=70%)", fontweight="bold")
axB.set_title("(B) Same indices, CCC — note Corema collapses here\n"
              "(low spatial variance deflates CCC, not drift)", fontweight="bold")
axB.legend(fontsize=9)

# shared band-basis legend
from matplotlib.patches import Patch
basis_handles = [Patch(color=c, label=b) for b, c in BASIS_COLOR.items()]
fig.legend(handles=basis_handles, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.06), title="Band basis (tick-label colour)",
           fontsize=9)

out = os.path.join(OUTPUT_DIR, "Fig_MultiIndex_DriftRobustness.png")
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out}")

# --- 6. Print the rRMSE-based ranking for the text ----------------------------
print("\nIndex ranking by median rRMSE (most -> least drift-robust):")
for rank, ix in enumerate(order, 1):
    print(f"  {rank}. {ix:11s} median_rRMSE={rrmse_med[ix]:7.2f}%  "
          f"basis={BAND_BASIS[ix]}")


---
## 6  Cross-Entity Statistical Comparison

**Part A — Acquisition geometry** (Corema album vs Avocado/Mango combined):
View Angle, True Solar Time, Solar Zenith. Avocado and Mango are merged here
because they share identical Planet acquisitions.

**Part B — Spectral/pixel characteristics** (3 entities):
Median NDVI, Veg_Fraction, and spectral band reflectances.
Pairwise Mann-Whitney U test + rank-biserial effect size.


In [ ]:
from scipy.stats import mannwhitneyu, ks_2samp
from itertools import combinations

# ── Part A: Geometry comparison (2 projects, avocado+mango combined) ──────────
print("=" * 72)
print("PART A — ACQUISITION GEOMETRY: Corema album vs Avocado/Mango")
print("=" * 72)

GEOM_VARS = {
    "View angle (deg)":      "View_Angle",
    "True Solar Time (h)":   "True_Solar_Hour",
    "Solar Zenith (deg)":    "Solar_Zenith",
}

proj_list   = list(PROJECTS.keys())
geom_rows   = []

for var_label, col in GEOM_VARS.items():

    samples = {
        p: df_images[df_images["Project"] == p][col].dropna().values
        for p in proj_list
    }

    for p in proj_list:
        s = samples[p]
        geom_rows.append({
            "Variable": var_label, "Group": p,
            "n": len(s),
            "Mean":   round(float(np.mean(s)), 3),
            "Median": round(float(np.median(s)), 3),
            "Std":    round(float(np.std(s)), 3),
            "P05":    round(float(np.percentile(s, 5)), 3),
            "P95":    round(float(np.percentile(s, 95)), 3),
        })

    pa, pb = proj_list[0], proj_list[1]
    u, p_mw  = mannwhitneyu(samples[pa], samples[pb], alternative="two-sided")
    ks, p_ks = ks_2samp(samples[pa], samples[pb])
    r_effect  = 1 - 2*u / (len(samples[pa]) * len(samples[pb]))
    print(f"  {var_label}: MW p={p_mw:.4f}  KS p={p_ks:.4f}  r={r_effect:.3f}")

df_geom_stats = pd.DataFrame(geom_rows)
display(df_geom_stats)


# ── Part B: Spectral comparison (3 entities) ──────────────────────────────────
print()
print("=" * 72)
print("PART B — SPECTRAL CHARACTERISTICS: 3-Entity Comparison")
print("=" * 72)

SPECTRAL_VARS = {
    "NDVI":         "NDVI",
    "NIR (SR)":     "NIR",
    "Red (SR)":     "Red",
    "Veg. fraction":"Veg_Fraction",
}

entity_list = list(ENTITIES.keys())
spec_rows   = []

for var_label, col in SPECTRAL_VARS.items():

    if col not in df_pixels.columns:
        print(f"  [SKIP] {var_label}: column not found"); continue

    samples = {
        e: df_pixels[df_pixels["Entity"] == e][col].dropna().values
        for e in entity_list
    }

    for e in entity_list:
        s = samples[e]
        spec_rows.append({
            "Variable": var_label, "Entity": e,
            "n": len(s),
            "Mean":   round(float(np.mean(s)), 4),
            "Median": round(float(np.median(s)), 4),
            "Std":    round(float(np.std(s)), 4),
        })

    # Pairwise significance tests
    for ea, eb in combinations(entity_list, 2):
        if len(samples[ea]) < 5 or len(samples[eb]) < 5:
            continue
        u, p_mw = mannwhitneyu(samples[ea], samples[eb], alternative="two-sided")
        r_eff   = 1 - 2*u / (len(samples[ea]) * len(samples[eb]))
        sig     = "***" if p_mw < 0.001 else ("**" if p_mw < 0.01 else ("*" if p_mw < 0.05 else "ns"))
        print(f"  {var_label:<18}  {ea} vs {eb:<16}  "
              f"p={p_mw:.4f} {sig}  r={r_eff:.3f}")

df_spec_stats = pd.DataFrame(spec_rows)
print()
display(df_spec_stats)

# Save both
all_stats = pd.concat([
    df_geom_stats.rename(columns={"Group": "Entity"}),
    df_spec_stats
], ignore_index=True)
out = os.path.join(OUTPUT_DIR, "Table_CrossEntity_Statistics.csv")
all_stats.to_csv(out, index=False)
print(f"\n[SAVED] {out}")


# ── Violin plot: spectral metrics, 3 entities ────────────────────────────────
spectral_plot = ["NDVI", "NIR", "Red"]
spectral_labs = ["NDVI", "NIR (SR)", "Red (SR)"]

fig, axes = plt.subplots(1, len(spectral_plot), figsize=(15, 6))

for ax, col, label in zip(axes, spectral_plot, spectral_labs):

    if col not in df_pixels.columns:
        ax.set_title(f"{label} (no data)"); continue

    data   = [df_pixels[df_pixels["Entity"]==e][col].dropna().values for e in entity_list]
    colors = [ENTITIES[e]["color"] for e in entity_list]

    parts = ax.violinplot(data, positions=range(len(entity_list)),
                          showmedians=True, showextrema=True)
    for pc, c in zip(parts["bodies"], colors):
        pc.set_facecolor(c); pc.set_alpha(0.55)
    parts["cmedians"].set_color("black"); parts["cmedians"].set_linewidth(2)

    ax.set_xticks(range(len(entity_list)))
    ax.set_xticklabels(entity_list, fontsize=10, rotation=15, ha="right")
    ax.set_ylabel(label, fontweight="bold")
    ax.set_title(label)

plt.suptitle("Cross-Entity Spectral Distributions (3 entities)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()

out_vio = os.path.join(OUTPUT_DIR, "Fig_CrossEntity_SpectralDistributions.png")
plt.savefig(out_vio, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out_vio}")


# ── CCC heatmap summary: entity x band ───────────────────────────────────────
band_order  = [l for _, l in BAND_COLS]
ccc_pivot   = df_ccc.pivot_table(index="Entity", columns="Band", values="CCC")
ccc_pivot   = ccc_pivot.reindex(columns=band_order, fill_value=np.nan)
ccc_pivot   = ccc_pivot.reindex(index=entity_list, fill_value=np.nan)

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(ccc_pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Lin's CCC")

ax.set_xticks(range(len(band_order)));  ax.set_xticklabels(band_order, rotation=40, ha="right")
ax.set_yticks(range(len(entity_list))); ax.set_yticklabels(entity_list, fontsize=11)

for i in range(len(entity_list)):
    for j in range(len(band_order)):
        val = ccc_pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color="black" if 0.35 < val < 0.85 else "white")

ax.set_title("Lin's CCC Heatmap — Entity x Spectral Band", fontweight="bold")
plt.tight_layout()

out_hm = os.path.join(OUTPUT_DIR, "Fig_CCC_Heatmap_EntityBand.png")
plt.savefig(out_hm, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out_hm}")


---
## Summary of outputs

All figures and tables are saved to:

```
03_figures_&_results/00_paper_comparison/
  Table2_DataQuality_Summary.csv
  Table_SpectralConcordance.csv
  Table_NDVI_Drift_Propagation.csv
  Table_CrossProject_Statistics.csv
  Fig4_ViewAngle_Comparison.png
  Fig5_OrbitalDrift_Comparison.png
  Fig6A_CCC_Corema_album.png
  Fig6B_CCC_Avocado_Mango.png
  Fig6C_CCC_CrossProject_Comparison.png
  Fig_NDVI_Drift_Sensitivity.png
  Fig_CrossProject_Distributions.png
```


In [ ]:
# # ============================================================
# # TO-DO #3: Mean fc of RETAINED avocado pixels (fc >= 70%)
# # For Table 3 of the paper
# # ============================================================
# VEG_FRACTION_MIN = 0.70

# for aoi in ['avocado', 'mango']:
#     pixels = pixel_maps[aoi]
#     retained = pixels[pixels['Veg_Fraction'] >= VEG_FRACTION_MIN]
#     print(f"\n{aoi.upper()}")
#     print(f"  Candidate pixels:        {len(pixels)}")
#     print(f"  Retained (fc >= 70%):    {len(retained)}")
#     print(f"  Retention rate:          {100*len(retained)/len(pixels):.1f}%")
#     print(f"  Mean fc of RETAINED:     {retained['Veg_Fraction'].mean():.3f}")
#     print(f"  Median fc of RETAINED:   {retained['Veg_Fraction'].median():.3f}")


# ============================================================
# TO-DO #3: Mean fc of RETAINED avocado pixels (fc >= 70%)
# For Table 3 of the paper
# ============================================================
VEG_FRACTION_MIN = 0.70

# Build pixel_maps: one row per unique pixel_id for each AOI
pixel_maps = {
    aoi: df_pixels[df_pixels["AOI"] == aoi].drop_duplicates("pixel_id")
    for aoi in ["avocado", "mango"]
}

for aoi in ["avocado", "mango"]:
    pixels = pixel_maps[aoi]
    retained = pixels[pixels["Veg_Fraction"] >= VEG_FRACTION_MIN]
    print(f"{aoi.upper()}")
    print(f"  Candidate pixels:        {len(pixels)}")
    print(f"  Retained (fc >= 70%):    {len(retained)}")
    print(f"  Retention rate:          {100*len(retained)/len(pixels):.1f}%")
    print(f"  Mean fc of RETAINED:     {retained['Veg_Fraction'].mean():.3f}")
    print(f"  Median fc of RETAINED:   {retained['Veg_Fraction'].median():.3f}")


In [ ]:
# ==============================================================================
# CONTROL: ¿el beneficio del filtro de pureza escala con la impureza de la cubierta?
# Cruza fc medio por entidad con la reduccion de rRMSE_NDVI al pasar de 0% -> 70%.
# n=3 cubiertas: ILUSTRATIVO, no concluyente. Reportar como "consistente con".
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os

rows = []
for ename in ENTITIES:
    r0  = df_sweep[(df_sweep["Entity"] == ename) & (df_sweep["Threshold_pct"] == 0)]
    r70 = df_sweep[(df_sweep["Entity"] == ename) & (df_sweep["Threshold_pct"] == 70)]
    if r0.empty or r70.empty:
        print(f"[skip] {ename}: falta fila 0% o 70% en df_sweep")
        continue
    rrmse_0  = float(r0["rrmse_NDVI"].values[0])
    rrmse_70 = float(r70["rrmse_NDVI"].values[0])
    fc_mean  = float(df_pixels[df_pixels["Entity"] == ename]["Veg_Fraction"].mean())
    rows.append({
        "Entity": ename,
        "fc_mean": round(fc_mean, 3),
        "rRMSE_NDVI_all": round(rrmse_0, 3),
        "rRMSE_NDVI_pure": round(rrmse_70, 3),
        "improvement_abs": round(rrmse_0 - rrmse_70, 3),
        "improvement_pct": round((rrmse_0 - rrmse_70) / rrmse_0 * 100, 1),
    })

df_scale = pd.DataFrame(rows).sort_values("fc_mean")
print("=" * 70)
print("FILTER BENEFIT vs COVER PURITY  (lower fc_mean -> expect larger benefit)")
print("=" * 70)
display(df_scale)

# Direccion de la relacion (Spearman, solo descriptivo con n=3)
if len(df_scale) >= 3:
    from scipy.stats import spearmanr
    rho, _ = spearmanr(df_scale["fc_mean"], df_scale["improvement_pct"])
    print(f"\nSpearman fc_mean vs improvement_pct: rho={rho:+.2f}  "
          f"(negativo = menor pureza -> mayor mejora, como se predice)")
    print("[!] n=3: ilustrativo, sin valor inferencial. Reportar como tendencia.")

# Mini-figura
fig, ax = plt.subplots(figsize=(6.5, 5))
for _, r in df_scale.iterrows():
    cfg = ENTITIES.get(r["Entity"], {})
    ax.scatter(r["fc_mean"], r["improvement_pct"], s=180,
               color=cfg.get("color", "grey"), marker=cfg.get("marker", "o"),
               edgecolor="black", zorder=3, label=r["Entity"])
    ax.annotate(r["Entity"], (r["fc_mean"], r["improvement_pct"]),
                textcoords="offset points", xytext=(8, 6), fontsize=9)
if len(df_scale) >= 2:
    z = np.polyfit(df_scale["fc_mean"], df_scale["improvement_pct"], 1)
    xs = np.linspace(df_scale["fc_mean"].min(), df_scale["fc_mean"].max(), 50)
    ax.plot(xs, np.polyval(z, xs), "--", color="grey", lw=1.2, alpha=0.8, zorder=1)
ax.set_xlabel("Mean fraction cover of the cover type (fc_mean)", fontweight="bold")
ax.set_ylabel("NDVI rRMSE reduction by fc>=70% filter (%)", fontweight="bold")
ax.set_title("Purity-filter benefit increases as cover gets less pure\n"
             "(n=3 covers — illustrative)", fontweight="bold", fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join(OUTPUT_DIR, "Fig_FilterBenefit_vs_Purity.png")
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(f"\n[SAVED] {out}")
df_scale.to_csv(os.path.join(OUTPUT_DIR, "Table_FilterBenefit_vs_Purity.csv"), index=False)
print("[SAVED] Table_FilterBenefit_vs_Purity.csv")

In [ ]:
# ==============================================================================
# CONTROL: ¿el beneficio del filtro de pureza escala con la impureza de la cubierta?
# Cruza fc medio por entidad con la reduccion de rRMSE_NDVI al pasar de 0% -> 70%.
# n=3 cubiertas: ILUSTRATIVO, no concluyente. Reportar como "consistente con".
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os

rows = []
for ename in ENTITIES:
    r0  = df_sweep[(df_sweep["Entity"] == ename) & (df_sweep["Threshold_pct"] == 0)]
    r70 = df_sweep[(df_sweep["Entity"] == ename) & (df_sweep["Threshold_pct"] == 70)]
    if r0.empty or r70.empty:
        print(f"[skip] {ename}: falta fila 0% o 70% en df_sweep")
        continue
    rrmse_0  = float(r0["rrmse_NDVI"].values[0])
    rrmse_70 = float(r70["rrmse_NDVI"].values[0])
    fc_mean  = float(df_pixels[df_pixels["Entity"] == ename]["Veg_Fraction"].mean())
    rows.append({
        "Entity": ename,
        "fc_mean": round(fc_mean, 3),
        "rRMSE_NDVI_all": round(rrmse_0, 3),
        "rRMSE_NDVI_pure": round(rrmse_70, 3),
        "improvement_abs": round(rrmse_0 - rrmse_70, 3),
        "improvement_pct": round((rrmse_0 - rrmse_70) / rrmse_0 * 100, 1),
    })

df_scale = pd.DataFrame(rows).sort_values("fc_mean")
print("=" * 70)
print("FILTER BENEFIT vs COVER PURITY  (lower fc_mean -> expect larger benefit)")
print("=" * 70)
display(df_scale)

# Direccion de la relacion (Spearman, solo descriptivo con n=3)
if len(df_scale) >= 3:
    from scipy.stats import spearmanr
    rho, _ = spearmanr(df_scale["fc_mean"], df_scale["improvement_pct"])
    print(f"\nSpearman fc_mean vs improvement_pct: rho={rho:+.2f}  "
          f"(negativo = menor pureza -> mayor mejora, como se predice)")
    print("[!] n=3: ilustrativo, sin valor inferencial. Reportar como tendencia.")

# Mini-figura
fig, ax = plt.subplots(figsize=(6.5, 5))
for _, r in df_scale.iterrows():
    cfg = ENTITIES.get(r["Entity"], {})
    ax.scatter(r["fc_mean"], r["improvement_pct"], s=180,
               color=cfg.get("color", "grey"), marker=cfg.get("marker", "o"),
               edgecolor="black", zorder=3, label=r["Entity"])
    ax.annotate(r["Entity"], (r["fc_mean"], r["improvement_pct"]),
                textcoords="offset points", xytext=(8, 6), fontsize=9)
# if len(df_scale) >= 2:
#     z = np.polyfit(df_scale["fc_mean"], df_scale["improvement_pct"], 1)
#     xs = np.linspace(df_scale["fc_mean"].min(), df_scale["fc_mean"].max(), 50)
#     ax.plot(xs, np.polyval(z, xs), "--", color="grey", lw=1.2, alpha=0.8, zorder=1)
ax.set_xlabel("Mean fraction cover of the cover type (fc_mean)", fontweight="bold")
ax.set_ylabel("NDVI rRMSE reduction by fc>=70% filter (%)", fontweight="bold")
# ax.set_title("Purity-filter benefit increases as cover gets less pure\n"
#              "(n=3 covers — illustrative)", fontweight="bold", fontsize=11)
ax.set_title("Purity-filter benefit concentrates in the spectrally mixed cover\n"
             "(Mango); pure covers unchanged (n=3 — illustrative)",
             fontweight="bold", fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join(OUTPUT_DIR, "Fig_FilterBenefit_vs_Purity.png")
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(f"\n[SAVED] {out}")
df_scale.to_csv(os.path.join(OUTPUT_DIR, "Table_FilterBenefit_vs_Purity.csv"), index=False)
print("[SAVED] Table_FilterBenefit_vs_Purity.csv")

In [ ]:
# ==============================================================================
# PHASE ANGLE (ángulo de dispersión) — re-lee 3 ángulos extra del XML ya indexado
# No toca el caché ni parse_planet_xml_raw. Usa df_meta (XML_Path) de la celda 4B.
# ==============================================================================
import os, numpy as np, pandas as pd
import xml.etree.ElementTree as ET

# NS ya está definido arriba en el notebook; lo redefino por seguridad si se ejecuta suelto
NS = {
    'gml': 'http://www.opengis.net/gml',
    'ps':  'http://schemas.planet.com/ps/v1/planet_product_metadata_geocorrected_level',
    'eop': 'http://earth.esa.int/eop',
    'opt': 'http://earth.esa.int/opt',
}

def _get_geometry_angles(xml_path):
    """Devuelve los 4 ángulos necesarios para el phase angle, desde el XML."""
    if not isinstance(xml_path, str) or not os.path.exists(xml_path):
        return (np.nan, np.nan, np.nan, np.nan)
    try:
        root = ET.parse(xml_path).getroot()
        sun_elev = float(root.find('.//opt:illuminationElevationAngle', NS).text)
        sun_az   = float(root.find('.//opt:illuminationAzimuthAngle',   NS).text)
        view_z   = float(root.find('.//ps:spaceCraftViewAngle',         NS).text)
        view_az  = float(root.find('.//ps:azimuthAngle',                NS).text)
        return (sun_elev, sun_az, view_z, view_az)
    except Exception:
        return (np.nan, np.nan, np.nan, np.nan)

def scattering_phase_angle(sun_elev_deg, sun_az_deg, view_zenith_deg, view_az_deg):
    sz = np.radians(90.0 - sun_elev_deg)
    vz = np.radians(view_zenith_deg)
    raz = np.radians(sun_az_deg - view_az_deg)
    cos_psi = np.cos(sz)*np.cos(vz) + np.sin(sz)*np.sin(vz)*np.cos(raz)
    return np.degrees(np.arccos(np.clip(cos_psi, -1.0, 1.0)))

# --- 1. Extraer ángulos para cada XML único de df_meta ------------------------
geo = df_meta[["Smart_ID", "XML_Path"]].drop_duplicates("Smart_ID").copy()
angs = geo["XML_Path"].apply(_get_geometry_angles)
geo[["Sun_Elev", "Sun_Az", "View_Z", "View_Az"]] = pd.DataFrame(
    angs.tolist(), index=geo.index)

geo["Phase_Angle"] = scattering_phase_angle(
    geo["Sun_Elev"], geo["Sun_Az"], geo["View_Z"], geo["View_Az"])

n_ok = geo["Phase_Angle"].notna().sum()
print(f"Escenas con phase angle calculado: {n_ok} / {len(geo)}")
print(geo[["Smart_ID","Sun_Elev","Sun_Az","View_Z","View_Az","Phase_Angle"]]
      .head(10).to_string(index=False))

# --- 2. Cruce con df_pairs_diag (Sección 4C): ¿el sesgo sigue al phase angle? --
if "df_pairs_diag" in dir() and not df_pairs_diag.empty:
    pa = geo.set_index("Smart_ID")["Phase_Angle"].to_dict()
    df_pairs_diag["d_phase"] = (
        df_pairs_diag["Img_late"].map(pa) - df_pairs_diag["Img_early"].map(pa)).abs()
    matched = df_pairs_diag["d_phase"].notna().sum()
    print(f"\nPares con Δphase calculable: {matched} / {len(df_pairs_diag)}")
    print("\nΔphase mediano por nivel (grados):")
    print(df_pairs_diag.groupby("Level")["d_phase"].median().round(3).to_string())
    print("\n[Esperado] N3 (sat+drift) deberia tener mayor Δphase que N2 (sat, misma hora)")
else:
    print("\n[i] df_pairs_diag no disponible; corre antes la Seccion 4C.")

# --- 3. Guardar para uso posterior --------------------------------------------
geo.to_csv(os.path.join(OUTPUT_DIR, "Table_SceneGeometry_PhaseAngle.csv"), index=False)
print("\n[SAVED] Table_SceneGeometry_PhaseAngle.csv")

In [ ]:
# ==============================================================================
# MECANISMO: ¿el sesgo de banda intra-par escala con el cambio de phase angle?
# Recorre df_pairs_diag (tiene Img_early/late + d_phase), calcula diff de banda
# por par, y correlaciona |diff_banda| vs d_phase. Prueba directa del mecanismo.
# ==============================================================================
import os, numpy as np, pandas as pd
from scipy.stats import spearmanr

INTERSAT_BANDS = ["Coastal_Blue", "Blue", "Green_I", "Green", "Yellow",
                  "Red", "RedEdge", "NIR"]
MIN_COMMON_PX = 5

# pre-index df_pixels por (Project, AOI, Date) para velocidad
day_groups = {k: v for k, v in df_pixels.groupby(["Project", "AOI", "Date_only"])}

rows = []
for _, r in df_pairs_diag.iterrows():
    if pd.isna(r.get("d_phase")):
        continue
    day = day_groups.get((r["Project"], r["AOI"], r["Date"]))
    if day is None:
        continue
    e = day[day["Smart_ID"] == r["Img_early"]].set_index("pixel_id")
    l = day[day["Smart_ID"] == r["Img_late"]].set_index("pixel_id")
    common = e.index.intersection(l.index)
    if len(common) < MIN_COMMON_PX:
        continue
    rec = {"Level": r["Level"], "d_phase": float(r["d_phase"]),
           "dt_min": float(r["dt_min"])}
    for b in INTERSAT_BANDS:
        if b in e.columns and b in l.columns:
            xe = e.loc[common, b].to_numpy(float)
            xl = l.loc[common, b].to_numpy(float)
            mu = np.nanmean(xe)
            rec[b] = abs(float(np.nanmedian(xl - xe) / mu * 100)) if mu > 1e-9 else np.nan
    rows.append(rec)

df_mech = pd.DataFrame(rows)
print(f"Pares con banda + d_phase: {len(df_mech)}")

# --- Correlación |diff_banda| vs d_phase, por banda ---------------------------
print("\n" + "=" * 70)
print("CORRELACIÓN  |sesgo de banda (%)|  vs  Δphase (deg)   [todos los pares]")
print("=" * 70)
corr_rows = []
for b in INTERSAT_BANDS:
    sub = df_mech[["d_phase", b]].dropna()
    if len(sub) < 20:
        continue
    rho, p = spearmanr(sub["d_phase"], sub[b])
    corr_rows.append({"Band": b, "n": len(sub), "Spearman_rho": round(rho, 3),
                      "p_value": p})
df_corr = pd.DataFrame(corr_rows)
print(df_corr.to_string(index=False))

# --- Control: ¿correlaciona mejor con d_phase o con dt? -----------------------
print("\nComparación: ¿geometría (Δphase) o tiempo (Δt) explica mejor el sesgo?")
print("(rho de cada banda con cada predictor; mayor |rho| = mejor predictor)")
cmp_rows = []
for b in INTERSAT_BANDS:
    sub = df_mech[["d_phase", "dt_min", b]].dropna()
    if len(sub) < 20:
        continue
    rho_ph, _ = spearmanr(sub["d_phase"], sub[b])
    rho_dt, _ = spearmanr(sub["dt_min"], sub[b])
    cmp_rows.append({"Band": b, "rho_vs_phase": round(rho_ph, 3),
                     "rho_vs_dt": round(rho_dt, 3),
                     "phase_wins": abs(rho_ph) > abs(rho_dt)})
df_cmp = pd.DataFrame(cmp_rows)
print(df_cmp.to_string(index=False))

df_corr.to_csv(os.path.join(OUTPUT_DIR, "Table_BandBias_vs_PhaseAngle.csv"), index=False)
print("\n[SAVED] Table_BandBias_vs_PhaseAngle.csv")

In [ ]:
# ==============================================================================
# DIAGNÓSTICO: por qué Tabla1 (80 pares Axarquía) ≠ Tabla2 (39+39=78)
# ==============================================================================
import pandas as pd

# Pares intradía a nivel SITIO (criterio Tabla 1: Δt>=40min, sin separar entidad)
# Reconstruye desde df_images como en la celda 4B/21
def site_pairs(df_img):
    recs = []
    for (proj, date), g in df_img.groupby(["Project", "Date_only"]):
        imgs = g.sort_values("True_Solar_Hour")[["Smart_ID","True_Solar_Hour"]].drop_duplicates()
        if len(imgs) < 2:
            continue
        delta = imgs["True_Solar_Hour"].iloc[-1] - imgs["True_Solar_Hour"].iloc[0]
        if delta >= 40/60:
            recs.append({"Project": proj, "Date": date, "delta_h": delta})
    return pd.DataFrame(recs)

sp = site_pairs(df_images)
print("PARES A NIVEL SITIO (criterio Tabla 1):")
print(sp.groupby("Project").size().to_string())

# Pares a nivel ENTIDAD (los que realmente usa la regresión / Tabla 2)
# Esto depende de tu df de pares por entidad; ajusta el nombre si es otro
print("\nPARES A NIVEL ENTIDAD (Tabla 2):")
for ent in ["Corema album", "Avocado", "Mango"]:
    sub = df_pixels[df_pixels["Entity"] == ent]
    n = 0
    for (aoi, date), day in sub.groupby(["AOI", "Date_only"]):
        imgs = day.groupby("Smart_ID")["True_Solar_Hour"].first().sort_values()
        if len(imgs) < 2:
            continue
        if imgs.iloc[-1] - imgs.iloc[0] < 40/60:
            continue
        e = day[day["Smart_ID"]==imgs.index[0]].set_index("pixel_id")
        l = day[day["Smart_ID"]==imgs.index[-1]].set_index("pixel_id")
        if len(e.index.intersection(l.index)) >= 5:
            n += 1
    print(f"  {ent:14s}: {n} pares")

print("\n[i] Si Axarquía sitio=80 pero avocado+mango=78, los 2 perdidos son")
print("    fechas con par válido a nivel escena pero <5 píxeles comunes en")
print("    NINGUNA de las dos entidades tras el filtro fc>=70%, o pares donde")
print("    el QC |ΔNDVI|<=0.40 descartó el par en ambas entidades.")

In [ ]:
# ¿El QC |ΔNDVI|<=0.40 descarta 1 par en avocado y mango? (eso explicaría 40->39)
for ent in ["Avocado", "Mango"]:
    sub = df_pixels[df_pixels["Entity"] == ent]
    n_sin_qc, n_con_qc = 0, 0
    for (aoi, date), day in sub.groupby(["AOI", "Date_only"]):
        imgs = day.groupby("Smart_ID")["True_Solar_Hour"].first().sort_values()
        if len(imgs) < 2 or imgs.iloc[-1]-imgs.iloc[0] < 40/60:
            continue
        e = day[day["Smart_ID"]==imgs.index[0]].set_index("pixel_id")
        l = day[day["Smart_ID"]==imgs.index[-1]].set_index("pixel_id")
        common = e.index.intersection(l.index)
        if len(common) < 5:
            continue
        n_sin_qc += 1
        dndvi = (l.loc[common,"NDVI"] - e.loc[common,"NDVI"]).median()
        if abs(dndvi) <= 0.40:
            n_con_qc += 1
    print(f"{ent}: sin QC={n_sin_qc}  con QC |ΔNDVI|<=0.40={n_con_qc}  (descarta {n_sin_qc-n_con_qc})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# ============================================================================
# FIGURA inter-satélite / phase angle — data-driven desde df_pairs_diag y df_corr
# ============================================================================

# ---- PANEL A: ΔΨ mediano por nivel (de df_pairs_diag) ----
level_order = ["N1_sameSat_short", "N2_diffSat_short", "N3_diffSat_long"]
level_labels = {
    "N1_sameSat_short": "N1\nsame sat\n≤10 min",
    "N2_diffSat_short": "N2\ndiff sat\n≤10 min",
    "N3_diffSat_long":  "N3\ndiff sat\n≥40 min",
}
level_colors = ["#888780", "#EF9F27", "#D85A30"]

dpsi_med = (df_pairs_diag[df_pairs_diag["d_phase"].notna()]
            .groupby("Level")["d_phase"].median())
levels = [l for l in level_order if l in dpsi_med.index]
dpsi   = [float(dpsi_med[l]) for l in levels]
labels = [level_labels[l] for l in levels]

# ---- PANEL B: correlación banda vs ΔΨ (de df_corr) ----
# Orden de presentación: visible-corto primero, NIR-anchored después.
# Blue y Green se excluyen (no se interpretan).
band_order   = ["Coastal_Blue", "Yellow", "Green_I", "NIR", "RedEdge", "Red"]
band_display = {"Coastal_Blue":"Coastal Blue","Yellow":"Yellow","Green_I":"Green I",
                "NIR":"NIR","RedEdge":"RedEdge","Red":"Red"}
visible_short = {"Coastal_Blue", "Yellow", "Green_I"}

corr_lookup = df_corr.set_index("Band")["Spearman_rho"].to_dict()
bands = [b for b in band_order if b in corr_lookup]
rho   = [float(corr_lookup[b]) for b in bands]
band_colors = ["#D85A30" if b in visible_short else "#888780" for b in bands]
band_names  = [band_display[b] for b in bands]

# ---- PLOT ----
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5))

xa = np.arange(len(levels))
axA.bar(xa, dpsi, color=level_colors[:len(levels)], edgecolor="black", linewidth=0.6, width=0.6)
for i, v in enumerate(dpsi):
    axA.text(i, v + max(dpsi)*0.02, f"{v:.2f}°", ha="center", va="bottom",
             fontsize=11, fontweight="bold")
axA.set_xticks(xa); axA.set_xticklabels(labels, fontsize=9)
axA.set_ylabel("Median ΔΨ  (degrees)", fontweight="bold")
axA.set_ylim(0, max(dpsi)*1.15)
axA.set_title("(A) Scattering geometry by level", fontweight="bold")
axA.spines[["top","right"]].set_visible(False)

yb = np.arange(len(bands))[::-1]
axB.barh(yb, rho, color=band_colors, edgecolor="black", linewidth=0.6, height=0.62)
for y, v in zip(yb, rho):
    axB.text(v + 0.012, y, f"{v:.2f}", va="center", fontsize=10)
axB.set_yticks(yb); axB.set_yticklabels(band_names, fontsize=10)
axB.set_xlabel("Spearman ρ  (|band bias| vs ΔΨ),  n = 1,752", fontweight="bold")
axB.set_xlim(0, max(rho)*1.18)
axB.set_title("(B) Band bias vs geometry", fontweight="bold")
axB.spines[["top","right"]].set_visible(False)

from matplotlib.patches import Patch
axB.legend(handles=[Patch(facecolor="#D85A30", edgecolor="black", label="short-visible (geometry-driven)"),
                    Patch(facecolor="#888780", edgecolor="black", label="NIR-anchored (resistant)")],
           fontsize=8, loc="lower right", framealpha=0.9)

plt.tight_layout()
out = os.path.join(OUTPUT_DIR, "Fig_InterSatellite_PhaseAngle.png")
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out}")

# verificación de coherencia con el texto
print("\nValores leídos (deben coincidir con Tabla X y el texto):")
for l, v in zip(levels, dpsi):
    print(f"  {l}: ΔΨ mediano = {v:.3f}°")
for b, r in zip(bands, rho):
    print(f"  {b}: ρ = {r:.3f}")